# TA-VA Analysis
## 1 Load model level data

In [1]:
from scipy import stats
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
import re
import pandasql as ps
from scipy.stats import wilcoxon
import pandas as pd

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Specify the experiment name and date to analyze
# Set experiment_name to None to analyze all experiments
experiment_name = "monitoring_vs_goal_based_analysis"
experiment_date = "20251108"  # Format: YYYYMMDD
# Note: The actual data is in the directory: consent_first_vs_goal_first_full_analysis_20251013

In [2]:
def extract_experiment_info(config_filename):
    """Extract experiment name and configuration from config filename.
    
    Example: consent_or_goal_sensitivity_analysis_(seed_2)_seed_2:_0-1000-0_20251013_165148_config.json
    Returns: ('consent_or_goal_sensitivity_analysis', '0-1000-0', '2')
    """
    # Remove _config.json suffix
    name = config_filename.replace('_config.json', '')
    
    # Pattern: {experiment_name}_(seed_{N})_seed_{N}:_{agent_config}_{timestamp}
    # Match the experiment name (everything before _(seed_)
    match = re.match(r'(.+?)_\(seed_(\d+)\)_seed_\2:_(.+?)_(\d{8}_\d{6})$', name)
    
    if match:
        exp_name = match.group(1)
        seed = match.group(2)
        agent_config = match.group(3)
        timestamp_date = match.group(4).split('_')[0]
        return exp_name, agent_config, seed, timestamp_date
    
    return None, None, None

def create_figures_directory(experiment_name, experiment_date):
    """Create figures directory for the experiment if it doesn't exist."""
    results_dir = Path("/Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results")
    
    # Find the experiment directory (it might have a different name than expected)
    experiment_dir = None
    for subdir in results_dir.iterdir():
        if subdir.is_dir():
            # Check if this directory contains files matching our experiment name and date
            configs_dir = subdir / "configs"
            if configs_dir.exists():
                for config_file in configs_dir.glob("*.json"):
                    exp_name, agent_config, seed, file_date = extract_experiment_info(config_file.name)
                    if exp_name == experiment_name and file_date == experiment_date:
                        experiment_dir = subdir
                        break
                if experiment_dir:
                    break
    
    if experiment_dir:
        figures_dir = experiment_dir / "figures"
        figures_dir.mkdir(exist_ok=True)
        return figures_dir
    else:
        # Fallback: create in main results directory
        figures_dir = results_dir / "figures"
        figures_dir.mkdir(exist_ok=True)
        return figures_dir

def load_simulation_data(experiment_name=None, experiment_date=None):
    """Load all simulation data and extract agent ratios."""
    results_dir = Path("/Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results")
    
    # Find all config files - check both main directories and experiment-specific subdirectories
    config_files = []
    
    # Check main configs directory
    main_configs_dir = results_dir / "configs"
    if main_configs_dir.exists():
        config_files.extend(list(main_configs_dir.glob("*.json")))
    
    # Check experiment-specific subdirectories
    if experiment_name and experiment_date:
        exp_subdir = results_dir / f"{experiment_name}_{experiment_date}"
        if exp_subdir.exists():
            exp_configs_dir = exp_subdir / "configs"
            if exp_configs_dir.exists():
                config_files.extend(list(exp_configs_dir.glob("*.json")))
                print(f"Found experiment-specific configs in: {exp_configs_dir}")
    
    # Also search all subdirectories for files that match the experiment name and date
    if experiment_name and experiment_date:
        for subdir in results_dir.iterdir():
            if subdir.is_dir():
                exp_configs_dir = subdir / "configs"
                if exp_configs_dir.exists():
                    # Check if any files in this directory match our criteria
                    matching_files = []
                    for config_file in exp_configs_dir.glob("*.json"):
                        exp_name, agent_config, seed, file_date = extract_experiment_info(config_file.name)
                        if exp_name == experiment_name and file_date == experiment_date:
                            matching_files.append(config_file)
                    
                    if matching_files:
                        config_files.extend(matching_files)
                        print(f"Found matching configs in subdirectory: {exp_configs_dir} ({len(matching_files)} files)")
    
    simulation_data = []
    timestamp_date = None
    
    for config_file in config_files:
        # Extract experiment info from filename
        exp_name, agent_config, seed, timestamp_date = extract_experiment_info(config_file.name)
        
        # Skip if we can't parse the filename or if it doesn't match the desired experiment
        if exp_name is None:
            print(f"Warning: Could not parse filename: {config_file.name}")
            continue
        
        if experiment_name is not None and exp_name != experiment_name:
            continue
        
        # Filter by experiment date if specified
        if experiment_date is not None and timestamp_date != experiment_date:
            continue
        
        # Load config
        with open(config_file, 'r') as f:
            config = json.load(f)
        
        # Extract agent counts
        params = config['parameters']
        monitoring = params.get('MonitoringAgent_COUNT', 0)
        goal_first = params.get('GoalFirstAgent_COUNT', 0)
        fifty_fifty = params.get('FiftyFiftyAgent_COUNT', 0)
        total_agents = monitoring + goal_first + fifty_fifty
        
        # Calculate ratios
        monitoring_ratio = monitoring / total_agents if total_agents > 0 else 0
        goal_ratio = goal_first / total_agents if total_agents > 0 else 0
        fifty_fifty_ratio = fifty_fifty / total_agents if total_agents > 0 else 0
        
        # Find corresponding model data file
        config_name = config_file.stem
        prefix = config_name.rsplit('_', 1)[0]
        
        # Look for data files in multiple locations
        model_file = None
        agent_file = None
        
        # List of directories to check for data files
        data_dirs_to_check = []
        
        # Check main data directory first
        main_data_dir = results_dir / "data"
        if main_data_dir.exists():
            data_dirs_to_check.append(main_data_dir)
        
        # Check experiment-specific subdirectory
        if experiment_name and experiment_date:
            exp_subdir = results_dir / f"{experiment_name}_{experiment_date}"
            if exp_subdir.exists():
                exp_data_dir = exp_subdir / "data"
                if exp_data_dir.exists():
                    data_dirs_to_check.append(exp_data_dir)
        
        # Also search all subdirectories for data files that match the experiment name and date
        if experiment_name and experiment_date:
            for subdir in results_dir.iterdir():
                if subdir.is_dir():
                    exp_data_dir = subdir / "data"
                    if exp_data_dir.exists() and exp_data_dir not in data_dirs_to_check:
                        # Check if any files in this directory match our criteria
                        # We'll check by looking for files with the same prefix as our config file
                        data_dirs_to_check.append(exp_data_dir)
        
        # Find the first directory that contains the required files
        for data_dir in data_dirs_to_check:
            model_file = data_dir / f"{prefix}_model.csv"
            agent_file = data_dir / f"{prefix}_agents.csv"
            if model_file.exists() and agent_file.exists():
                break
        
        if model_file and model_file.exists():
            # Load model data
            model_df = pd.read_csv(model_file)
            agent_df = pd.read_csv(agent_file)

            # Calculate CI state ratios
            # Handle division by zero
            model_df["Consent Violation Ratio"] = model_df["Total Violated Consents"] / model_df["Total Consent Activations"].replace(0, np.nan)
            model_df["Consent Fulfillment Ratio"] = model_df["Total Fulfilled Consents"] / model_df["Total Consent Activations"].replace(0, np.nan)
            model_df["Consent Unrealized Ratio"] = model_df["Total Unrealized Consents"] / model_df["Total Consent Activations"].replace(0, np.nan)
            model_df["Consent Deferred Ratio"] = model_df["Total Deferred Consents"] / model_df["Total Consent Activations"].replace(0, np.nan)
            model_df["Resource Conflict Counter Goal Accomplishment Ratio"] = model_df["Total Resource Conflicts"] / model_df["Total Resource Conflict Accomplished Counter Goals"].replace(0, np.nan)
            
            # Exclude the last early_stop_steps - 1 steps before getting final values
            # But here we should also check if no additional goals were really accomplished after the early stop steps.
            early_stop_steps = config.get('early_stop_steps', 0)
            if early_stop_steps > 1:
                # Exclude the last (early_stop_steps - 1) rows
                steps_to_exclude = early_stop_steps - 1
                distinct_agent_count_q = """SELECT COUNT(DISTINCT AgentID) as AGENT_COUNT FROM agent_df WHERE Step = 1"""
                distinct_agent_count = ps.sqldf(distinct_agent_count_q, locals())["AGENT_COUNT"][0]
                if len(model_df) > steps_to_exclude and model_df.iloc[-steps_to_exclude]['Total Accomplished Goals'] == model_df.iloc[-1]['Total Accomplished Goals']:
                    model_df = model_df.iloc[:-steps_to_exclude]
                    agent_df = agent_df.iloc[:-steps_to_exclude*distinct_agent_count]
                    
            
            # Get final values (last distinct_agent_count rows after exclusion)
            final_agent_values = agent_df.iloc[-distinct_agent_count:]
            
            # Calculate agent-level metrics for each agent type
            monitoring_mask = final_agent_values['Agent Persona'] == 'MonitoringAgent'
            goal_first_mask = final_agent_values['Agent Persona'] == 'GoalFirstAgent'
            
            # Calculate consent-related metrics from the available columns
            # Note: The CSV has different column names than expected
            avg_accomplished_goals_monitoring_agent = final_agent_values[monitoring_mask]['Accomplished Goals'].mean() if monitoring_mask.any() else 0
            avg_accomplished_goals_goal_first_agent = final_agent_values[goal_first_mask]['Accomplished Goals'].mean() if goal_first_mask.any() else 0
            avg_remaining_goals_monitoring_agent = final_agent_values[monitoring_mask]['Remaining Goals'].mean() if monitoring_mask.any() else 0
            avg_remaining_goals_goal_first_agent = final_agent_values[goal_first_mask]['Remaining Goals'].mean() if goal_first_mask.any() else 0
            avg_resource_conflicts_monitoring_agent = final_agent_values[monitoring_mask]['Resource Conflicts'].mean() if monitoring_mask.any() else 0
            avg_resource_conflicts_goal_first_agent = final_agent_values[goal_first_mask]['Resource Conflicts'].mean() if goal_first_mask.any() else 0
            avg_counter_goal_accomplishments_monitoring_agent = final_agent_values[monitoring_mask]['Counter Conflict Goal Accomplishments'].mean() if monitoring_mask.any() else 0
            avg_counter_goal_accomplishments_goal_first_agent = final_agent_values[goal_first_mask]['Counter Conflict Goal Accomplishments'].mean() if goal_first_mask.any() else 0
            
            # Calculate consent metrics from available columns
            # Separately for R (Receiver) and G (Giver) and agent type.
            total_consents_monitoring_r = final_agent_values[monitoring_mask]['Number of Consents as R'].mean() if monitoring_mask.any() else 0
            total_consents_monitoring_g = final_agent_values[monitoring_mask]['Number of Consents as G'].mean() if monitoring_mask.any() else 0
            total_consents_goal_first_r = final_agent_values[goal_first_mask]['Number of Consents as R'].mean() if goal_first_mask.any() else 0
            total_consents_goal_first_g = final_agent_values[goal_first_mask]['Number of Consents as G'].mean() if goal_first_mask.any() else 0
            violated_consents_monitoring_r = final_agent_values[monitoring_mask]['Number of Consents as R Violated'].mean() if monitoring_mask.any() else 0
            violated_consents_monitoring_g = final_agent_values[monitoring_mask]['Number of Consents as G Violated'].mean() if monitoring_mask.any() else 0
            violated_consents_goal_first_r = final_agent_values[goal_first_mask]['Number of Consents as R Violated'].mean() if goal_first_mask.any() else 0
            violated_consents_goal_first_g = final_agent_values[goal_first_mask]['Number of Consents as G Violated'].mean() if goal_first_mask.any() else 0
            
            fulfilled_consents_monitoring_r = final_agent_values[monitoring_mask]['Number of Consents as R Fulfilled'].mean() if monitoring_mask.any() else 0
            fulfilled_consents_monitoring_g = final_agent_values[monitoring_mask]['Number of Consents as G Fulfilled'].mean() if monitoring_mask.any() else 0
            fulfilled_consents_goal_first_r = final_agent_values[goal_first_mask]['Number of Consents as R Fulfilled'].mean() if goal_first_mask.any() else 0
            fulfilled_consents_goal_first_g = final_agent_values[goal_first_mask]['Number of Consents as G Fulfilled'].mean() if goal_first_mask.any() else 0
            
            # Calculate ratios (avoid division by zero)
            avg_consent_violation_ratio_monitoring_r = (violated_consents_monitoring_r / total_consents_monitoring_r) if total_consents_monitoring_r > 0 else 0
            avg_consent_violation_ratio_monitoring_g = (violated_consents_monitoring_g / total_consents_monitoring_g) if total_consents_monitoring_g > 0 else 0
            avg_consent_violation_ratio_goal_first_r = (violated_consents_goal_first_r / total_consents_goal_first_r) if total_consents_goal_first_r > 0 else 0
            avg_consent_violation_ratio_goal_first_g = (violated_consents_goal_first_g / total_consents_goal_first_g) if total_consents_goal_first_g > 0 else 0
            
            avg_consent_fulfillment_ratio_monitoring_r = (fulfilled_consents_monitoring_r / total_consents_monitoring_r) if total_consents_monitoring_r > 0 else 0
            avg_consent_fulfillment_ratio_monitoring_g = (fulfilled_consents_monitoring_g / total_consents_monitoring_g) if total_consents_monitoring_g > 0 else 0
            avg_consent_fulfillment_ratio_goal_first_r = (fulfilled_consents_goal_first_r / total_consents_goal_first_r) if total_consents_goal_first_r > 0 else 0
            avg_consent_fulfillment_ratio_goal_first_g = (fulfilled_consents_goal_first_g / total_consents_goal_first_g) if total_consents_goal_first_g > 0 else 0
        
            
            # Resource conflict counter goal accomplishment ratio
            avg_resource_conflict_counter_goal_accomplishment_ratio_monitoring_agent = (avg_resource_conflicts_monitoring_agent / avg_counter_goal_accomplishments_monitoring_agent) if avg_counter_goal_accomplishments_monitoring_agent > 0 else 0
            avg_resource_conflict_counter_goal_accomplishment_ratio_goal_first_agent = (avg_resource_conflicts_goal_first_agent / avg_counter_goal_accomplishments_goal_first_agent) if avg_counter_goal_accomplishments_goal_first_agent > 0 else 0
            
            # Calculate interaction and timing metrics
            avg_finished_step_monitoring_agent = final_agent_values[monitoring_mask]['Finished Step'].mean() if monitoring_mask.any() else 0
            avg_finished_step_goal_first_agent = final_agent_values[goal_first_mask]['Finished Step'].mean() if goal_first_mask.any() else 0
            avg_longest_idle_time_monitoring_agent = final_agent_values[monitoring_mask]['Longest Idle Time'].mean() if monitoring_mask.any() else 0
            avg_longest_idle_time_goal_first_agent = final_agent_values[goal_first_mask]['Longest Idle Time'].mean() if goal_first_mask.any() else 0
            avg_distinct_agents_interacted_r_monitoring_agent = final_agent_values[monitoring_mask]['Number of Distinct Agents Interacted as R'].mean() if monitoring_mask.any() else 0
            avg_distinct_agents_interacted_r_goal_first_agent = final_agent_values[goal_first_mask]['Number of Distinct Agents Interacted as R'].mean() if goal_first_mask.any() else 0
            avg_distinct_agents_interacted_g_monitoring_agent = final_agent_values[monitoring_mask]['Number of Distinct Agents Interacted as G'].mean() if monitoring_mask.any() else 0
            avg_distinct_agents_interacted_g_goal_first_agent = final_agent_values[goal_first_mask]['Number of Distinct Agents Interacted as G'].mean() if goal_first_mask.any() else 0
            # New: total idle time per agent
            avg_total_idle_time_monitoring_agent = final_agent_values[monitoring_mask]['Total Idle Time'].mean() if monitoring_mask.any() else 0
            avg_total_idle_time_goal_first_agent = final_agent_values[goal_first_mask]['Total Idle Time'].mean() if goal_first_mask.any() else 0
            
            # Calculate steps for this run as the last value of the Step/index column
            if not model_df.empty:
                if 'Step' in model_df.columns:
                    avg_steps_overall = int(pd.to_numeric(model_df['Step'], errors='coerce').dropna().iloc[-1])
                else:
                    first_col = model_df.columns[0]
                    avg_steps_overall = int(pd.to_numeric(model_df[first_col], errors='coerce').dropna().iloc[-1])
            else:
                avg_steps_overall = np.nan
            final_values = model_df.iloc[-1]
            
            simulation_data.append({
                'experiment_name': exp_name,
                'agent_config': agent_config,
                'seed': seed,
                'config_name': config_name,
                'monitoring_count': monitoring,
                'goal_first_count': goal_first,
                'fifty_fifty_count': fifty_fifty,
                'total_agents': total_agents,
                'accomplished_goals': final_values['Total Accomplished Goals'],
                'remaining_goals': final_values['Total Remaining Goals'],
                'violated_consents': final_values['Total Violated Consents'],
                'total_consents': final_values['Total Consent Activations'],
                'resource_conflicts': final_values['Total Resource Conflicts'],
                'counter_goal_accomplishments': final_values['Total Resource Conflict Accomplished Counter Goals'],
                'consent_violation_ratio': final_values['Consent Violation Ratio'],
                'consent_fulfillment_ratio': final_values['Consent Fulfillment Ratio'],
                'consent_unrealized_ratio': final_values['Consent Unrealized Ratio'],
                'consent_deferred_ratio': final_values['Consent Deferred Ratio'],
                'resource_conflict_counter_goal_accomplishment_ratio': final_values['Resource Conflict Counter Goal Accomplishment Ratio'],
                'max_steps': config.get('max_steps', 1000),
                'avg_steps_overall': avg_steps_overall,
                'avg_accomplished_goals_monitoring_agent': avg_accomplished_goals_monitoring_agent,
                'avg_accomplished_goals_goal_first_agent': avg_accomplished_goals_goal_first_agent,
                'avg_remaining_goals_monitoring_agent': avg_remaining_goals_monitoring_agent,
                'avg_remaining_goals_goal_first_agent': avg_remaining_goals_goal_first_agent,
                # R (Receiver) specific metrics
                'avg_total_consents_monitoring_r': total_consents_monitoring_r,
                'avg_total_consents_goal_first_r': total_consents_goal_first_r,
                'avg_violated_consents_monitoring_r': violated_consents_monitoring_r,
                'avg_violated_consents_goal_first_r': violated_consents_goal_first_r,
                'avg_fulfilled_consents_monitoring_r': fulfilled_consents_monitoring_r,
                'avg_fulfilled_consents_goal_first_r': fulfilled_consents_goal_first_r,
                'avg_consent_violation_ratio_monitoring_r': avg_consent_violation_ratio_monitoring_r,
                'avg_consent_violation_ratio_goal_first_r': avg_consent_violation_ratio_goal_first_r,
                'avg_consent_fulfillment_ratio_monitoring_r': avg_consent_fulfillment_ratio_monitoring_r,
                'avg_consent_fulfillment_ratio_goal_first_r': avg_consent_fulfillment_ratio_goal_first_r,
                
                # G (Giver) specific metrics
                'avg_total_consents_monitoring_g': total_consents_monitoring_g,
                'avg_total_consents_goal_first_g': total_consents_goal_first_g,
                'avg_violated_consents_monitoring_g': violated_consents_monitoring_g,
                'avg_violated_consents_goal_first_g': violated_consents_goal_first_g,
                'avg_fulfilled_consents_monitoring_g': fulfilled_consents_monitoring_g,
                'avg_fulfilled_consents_goal_first_g': fulfilled_consents_goal_first_g,
                'avg_consent_violation_ratio_monitoring_g': avg_consent_violation_ratio_monitoring_g,
                'avg_consent_violation_ratio_goal_first_g': avg_consent_violation_ratio_goal_first_g,
                'avg_consent_fulfillment_ratio_monitoring_g': avg_consent_fulfillment_ratio_monitoring_g,
                'avg_consent_fulfillment_ratio_goal_first_g': avg_consent_fulfillment_ratio_goal_first_g,
                
                # General agent metrics
                'avg_resource_conflicts_monitoring_agent': avg_resource_conflicts_monitoring_agent,
                'avg_resource_conflicts_goal_first_agent': avg_resource_conflicts_goal_first_agent,
                'avg_counter_goal_accomplishments_monitoring_agent': avg_counter_goal_accomplishments_monitoring_agent,
                'avg_counter_goal_accomplishments_goal_first_agent': avg_counter_goal_accomplishments_goal_first_agent,
                'avg_resource_conflict_counter_goal_accomplishment_ratio_monitoring_agent': avg_resource_conflict_counter_goal_accomplishment_ratio_monitoring_agent,
                'avg_resource_conflict_counter_goal_accomplishment_ratio_goal_first_agent': avg_resource_conflict_counter_goal_accomplishment_ratio_goal_first_agent,
                
                # Interaction and timing metrics
                'avg_finished_step_monitoring_agent': avg_finished_step_monitoring_agent,
                'avg_finished_step_goal_first_agent': avg_finished_step_goal_first_agent,
                'avg_longest_idle_time_monitoring_agent': avg_longest_idle_time_monitoring_agent,
                'avg_longest_idle_time_goal_first_agent': avg_longest_idle_time_goal_first_agent,
                'avg_distinct_agents_interacted_r_monitoring_agent': avg_distinct_agents_interacted_r_monitoring_agent,
                'avg_distinct_agents_interacted_r_goal_first_agent': avg_distinct_agents_interacted_r_goal_first_agent,
                'avg_distinct_agents_interacted_g_monitoring_agent': avg_distinct_agents_interacted_g_monitoring_agent,
                'avg_distinct_agents_interacted_g_goal_first_agent': avg_distinct_agents_interacted_g_goal_first_agent,
                # New: total idle time
                'avg_total_idle_time_monitoring_agent': avg_total_idle_time_monitoring_agent,
                'avg_total_idle_time_goal_first_agent': avg_total_idle_time_goal_first_agent,
            })
        else:
            print(f"Warning: Model data file not found for {config_name}")
    
    return pd.DataFrame(simulation_data), timestamp_date

def create_agent_ratio_analysis(experiment_name=None, experiment_date=None):
    """Create comprehensive analysis of how metrics change with agent ratios.
    
    This function averages results across all seeds for each experiment configuration.
    """
    print(f"Analyzing experiment: {experiment_name}, date: {experiment_date}")
    
    # Create figures directory
    figures_dir = create_figures_directory(experiment_name, experiment_date)
    
    # Load data
    df, timestamp_date = load_simulation_data(experiment_name=experiment_name, experiment_date=experiment_date)
    
    if df.empty:
        print("No simulation data found!")
        return
    
    # Group by experiment_name and agent_config, then calculate mean and std
    metrics_to_average = [
        'monitoring_count', 'goal_first_count', 'fifty_fifty_count', 'total_agents',
        'accomplished_goals', 'remaining_goals', 'violated_consents', 'total_consents',
        'resource_conflicts', 'counter_goal_accomplishments',
        'consent_violation_ratio', 'consent_fulfillment_ratio', 
        'consent_unrealized_ratio', 'consent_deferred_ratio',
        'resource_conflict_counter_goal_accomplishment_ratio', 'avg_steps_overall'
    ]
    
    # Calculate mean and standard error for each metric
    grouped = df.groupby(['experiment_name', 'agent_config'])
    
    mean_df = grouped[metrics_to_average].mean().reset_index()

    return mean_df, df

mean_df, df = create_agent_ratio_analysis(experiment_name="monitoring_vs_goal_based_analysis", experiment_date="20251108")


Analyzing experiment: monitoring_vs_goal_based_analysis, date: 20251108
Found experiment-specific configs in: /Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results/monitoring_vs_goal_based_analysis_20251108/configs
Found matching configs in subdirectory: /Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results/monitoring_vs_goal_based_analysis_20251108/configs (110 files)


In [3]:
df

,experiment_name,agent_config,seed,config_name,monitoring_count,goal_first_count,fifty_fifty_count,total_agents,accomplished_goals,remaining_goals,...,avg_finished_step_monitoring_agent,avg_finished_step_goal_first_agent,avg_longest_idle_time_monitoring_agent,avg_longest_idle_time_goal_first_agent,avg_distinct_agents_interacted_r_monitoring_agent,avg_distinct_agents_interacted_r_goal_first_agent,avg_distinct_agents_interacted_g_monitoring_agent,avg_distinct_agents_interacted_g_goal_first_agent,avg_total_idle_time_monitoring_agent,avg_total_idle_time_goal_first_agent
0,monitoring_vs_goal_based_analysis,0-500-0-500,13,monitoring_vs_goal_based_analysis_(seed_13)_se...,500,500,0,1000,2929.0,71.0,...,19.981520,21.002079,15.504000,16.778000,10.774000,7.716000,8.232000,10.258000,17.742000,19.074000
1,monitoring_vs_goal_based_analysis,0-100-0-900,2,monitoring_vs_goal_based_analysis_(seed_2)_see...,900,100,0,1000,2993.0,7.0,...,14.141268,15.632653,9.613333,11.340000,10.293333,7.870000,9.758889,12.680000,11.153333,12.830000
2,monitoring_vs_goal_based_analysis,0-600-0-400,456,monitoring_vs_goal_based_analysis_(seed_456)_s...,400,600,0,1000,2812.0,188.0,...,24.082667,24.485130,19.167500,20.925000,11.532500,7.485000,8.010000,9.833333,22.450000,23.733333
3,monitoring_vs_goal_based_analysis,0-1000-0-0,13,monitoring_vs_goal_based_analysis_(seed_13)_se...,0,1000,0,1000,524.0,2476.0,...,0.000000,4.423077,0.000000,10.093000,0.000000,3.441000,0.000000,3.441000,0.000000,10.305000
4,monitoring_vs_goal_based_analysis,0-500-0-500,35,monitoring_vs_goal_based_analysis_(seed_35)_se...,500,500,0,1000,2946.0,54.0,...,21.837398,22.458506,16.518000,17.698000,11.346000,7.748000,8.322000,10.772000,19.176000,20.202000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215,monitoring_vs_goal_based_analysis,0-900-0-100,35,monitoring_vs_goal_based_analysis_(seed_35)_se...,100,900,0,1000,660.0,2340.0,...,21.666667,15.396226,53.520000,55.026667,7.590000,4.011111,4.380000,4.367778,55.810000,56.722222
216,monitoring_vs_goal_based_analysis,0-900-0-100,13,monitoring_vs_goal_based_analysis_(seed_13)_se...,100,900,0,1000,1070.0,1930.0,...,73.045455,80.937107,182.270000,188.975556,14.430000,4.948889,5.490000,5.942222,192.750000,200.958889
217,monitoring_vs_goal_based_analysis,0-700-0-300,999,monitoring_vs_goal_based_analysis_(seed_999)_s...,300,700,0,1000,2353.0,647.0,...,27.198198,27.615711,27.043333,29.252857,11.396667,6.875714,7.203333,8.672857,31.186667,33.285714
218,monitoring_vs_goal_based_analysis,0-300-0-700,13,monitoring_vs_goal_based_analysis_(seed_13)_se...,700,300,0,1000,2992.0,8.0,...,15.638054,16.333333,10.877143,11.850000,10.371429,7.833333,8.885714,11.300000,12.662857,13.500000


##  Graph 01-02: 1-way ANOVA: Accomplished Goals

In [3]:
df_sorted = df.drop_duplicates().sort_values(by=['goal_first_count', 'seed'], ascending=True)
df_sorted["consent_violation_ratio"] = df_sorted["violated_consents"] / df_sorted["total_consents"]
groups = [
    df_sorted[df_sorted["goal_first_count"] == r]["accomplished_goals"]
    for r in sorted(df_sorted["goal_first_count"].unique())
]

# One-way ANOVA
F, p = stats.f_oneway(*groups)

# Effect size: eta-squared for one-way ANOVA
k = len(groups)                             # number of groups
ns = [len(g) for g in groups]
N = sum(ns)                                 # total sample size
df_between = k - 1
df_within = N - k
eta_sq = (F * df_between) / (F * df_between + df_within)

# Get min / max values of the averages graph
max_accomplished_goals = mean_df.groupby("goal_first_count")["accomplished_goals"].mean().max()
min_accomplished_goals = mean_df.groupby("goal_first_count")["accomplished_goals"].mean().min()

print(f"F: {F:.4f}, p: {p:.3e}")
print(f"Eta-squared (effect size): {eta_sq:.4f}")
print(f"Max: {max_accomplished_goals}, Min: {min_accomplished_goals}")




F: 702.0106, p: 3.535e-87
Eta-squared (effect size): 0.9861
Max: 3000.0, Min: 512.1


## Graph 03: 1-way ANOVA: Consent Violation Ratio

In [4]:
groups = [
    df_sorted[df_sorted["goal_first_count"] == r]["consent_violation_ratio"]
    for r in sorted(df_sorted["goal_first_count"].unique())
]

# One-way ANOVA
F, p = stats.f_oneway(*groups)

# Effect size: eta-squared for one-way ANOVA
k = len(groups)                             # number of groups
ns = [len(g) for g in groups]
N = sum(ns)                                 # total sample size
df_between = k - 1
df_within = N - k
eta_sq = (F * df_between) / (F * df_between + df_within)

max_consent_violation_ratio = mean_df.groupby("goal_first_count")["consent_violation_ratio"].mean().max()
min_consent_violation_ratio = mean_df.groupby("goal_first_count")["consent_violation_ratio"].mean().min()

print(f"F: {F:.4f}, p: {p:.3e}")
print(f"Eta-squared (effect size): {eta_sq:.4f}")
print(f"Max: {max_consent_violation_ratio}, Min: {min_consent_violation_ratio}")

F: 466.6374, p: 1.465e-78
Eta-squared (effect size): 0.9792
Max: 0.6756421965252705, Min: 0.1549459627001371


## Get Agent Level Data

In [5]:

def create_agent_level_analysis(experiment_name=None, experiment_date=None):
    """Create analysis of agent-level metrics comparing MonitoringAgent and GoalFirstAgent.
    
    This function shows how individual agent performance varies across different configurations.
    Uses agent CSV files and config JSON files directly, no model CSV files.
    """
    print(f"\nCreating Agent-Level Analysis for: {experiment_name}, date: {experiment_date}")
    
    # Create figures directory
    figures_dir = create_figures_directory(experiment_name, experiment_date)
    print(f"Figures will be saved to: {figures_dir}")
    
    results_dir = Path("/Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results")
    
    def _find_agent_file(prefix: str):
        main_data = results_dir / "data"
        p = main_data / f"{prefix}_agents.csv"
        if p.exists():
            return p
        for sub in results_dir.iterdir():
            if sub.is_dir():
                d = sub / "data"
                q = d / f"{prefix}_agents.csv"
                if d.exists() and q.exists():
                    return q
        return None
    
    def _find_model_file(prefix: str):
        main_data = results_dir / "data"
        p = main_data / f"{prefix}_model.csv"
        if p.exists():
            return p
        for sub in results_dir.iterdir():
            if sub.is_dir():
                d = sub / "data"
                q = d / f"{prefix}_model.csv"
                if d.exists() and q.exists():
                    return q
        return None
    
    # Find all config files - check both main directories and experiment-specific subdirectories
    config_files = []
    
    # Check main configs directory
    main_configs_dir = results_dir / "configs"
    if main_configs_dir.exists():
        config_files.extend(list(main_configs_dir.glob("*.json")))
    
    # Check experiment-specific subdirectories
    if experiment_name and experiment_date:
        exp_subdir = results_dir / f"{experiment_name}_{experiment_date}"
        if exp_subdir.exists():
            exp_configs_dir = exp_subdir / "configs"
            if exp_configs_dir.exists():
                config_files.extend(list(exp_configs_dir.glob("*.json")))
                print(f"Found experiment-specific configs in: {exp_configs_dir}")
    
    # Also search all subdirectories for files that match the experiment name and date
    if experiment_name and experiment_date:
        for subdir in results_dir.iterdir():
            if subdir.is_dir():
                exp_configs_dir = subdir / "configs"
                if exp_configs_dir.exists():
                    # Check if any files in this directory match our criteria
                    matching_files = []
                    for config_file in exp_configs_dir.glob("*.json"):
                        exp_name, agent_config, seed, file_date = extract_experiment_info(config_file.name)
                        if exp_name == experiment_name and file_date == experiment_date:
                            matching_files.append(config_file)
                    
                    if matching_files:
                        config_files.extend(matching_files)
                        print(f"Found matching configs in subdirectory: {exp_configs_dir} ({len(matching_files)} files)")
    
    # Collect agent-level data from all agent CSV files
    agent_data_list = []
    agent_data_list_all_steps = []
    
    for config_file in config_files:
        # Extract experiment info from filename
        exp_name, agent_config, seed, timestamp_date = extract_experiment_info(config_file.name)
        
        # Skip if we can't parse the filename or if it doesn't match the desired experiment
        if exp_name is None:
            print(f"Warning: Could not parse filename: {config_file.name}")
            continue
        
        if experiment_name is not None and exp_name != experiment_name:
            continue
        
        # Filter by experiment date if specified
        if experiment_date is not None and timestamp_date != experiment_date:
            continue
        
        # Load config to get agent counts
        try:
            with open(config_file, 'r') as f:
                config = json.load(f)
        except Exception as e:
            print(f"Warning: Could not load config file {config_file}: {e}")
            continue
        
        # Extract agent counts from config
        params = config.get('parameters', {})
        monitoring = params.get('MonitoringAgent_COUNT', 0)
        goal_first = params.get('GoalFirstAgent_COUNT', 0)
        fifty_fifty = params.get('FiftyFiftyAgent_COUNT', 0)
        total_agents = monitoring + goal_first + fifty_fifty
        
        # Get config name (without _config.json suffix)
        config_name = config_file.stem
        
        # Find corresponding agent file
        prefix = config_name.rsplit('_', 1)[0]
        agent_file = _find_agent_file(prefix)
        model_file = _find_model_file(prefix)
        
        if agent_file is None or not agent_file.exists():
            print(f"Warning: Agent file not found for {prefix}")
            continue
        
        if model_file is None or not model_file.exists():
            print(f"Warning: Model file not found for {prefix}")
            continue
        
        try:
            agent_df = pd.read_csv(agent_file)
            model_df = pd.read_csv(model_file)
            
            # Get the step column
            step_col = 'Step' if 'Step' in agent_df.columns else agent_df.columns[0]

            early_stop_steps = config.get('early_stop_steps', 0)
            if early_stop_steps > 1:
                # Exclude the last (early_stop_steps - 1) rows
                steps_to_exclude = early_stop_steps - 1
                distinct_agent_count_q = """SELECT COUNT(DISTINCT AgentID) as AGENT_COUNT FROM agent_df WHERE Step = 1"""
                distinct_agent_count = ps.sqldf(distinct_agent_count_q, locals())["AGENT_COUNT"][0]

                if len(model_df) > steps_to_exclude and model_df.iloc[-steps_to_exclude]['Total Accomplished Goals'] == model_df.iloc[-1]['Total Accomplished Goals']:
                    agent_df = agent_df.iloc[:-steps_to_exclude*distinct_agent_count]
            
            # Get final step and calculate avg_steps_overall from agent CSV
            steps = pd.to_numeric(agent_df[step_col], errors='coerce')
            last_step = steps.max()
            avg_steps_overall = int(last_step) if not pd.isna(last_step) else np.nan
            
            final_agent_values = agent_df[steps == last_step].copy()
            
            if 'Agent Persona' not in final_agent_values.columns:
                print(f"Warning: 'Agent Persona' column not found in {agent_file}")
                continue
            
            # Create masks for agent types
            monitoring_mask = final_agent_values['Agent Persona'] == 'MonitoringAgent'
            goal_first_mask = final_agent_values['Agent Persona'] == 'GoalFirstAgent'

            final_agent_values["seed"] = seed
            final_agent_values["agent_config"] = agent_config

            agent_df["seed"] = seed
            agent_df["agent_config"] = agent_config
            
            agent_data_list.append(final_agent_values)
            agent_data_list_all_steps.append(agent_df)
        except Exception as e:
            print(f"Error processing {prefix}: {e}")
            import traceback
            traceback.print_exc()
            continue

    if len(agent_data_list) == 0:
        print("Warning: No agent data collected. Returning None.")
        return None
    
    all_agent_values_df = pd.concat(agent_data_list)
    all_agent_values_df["goal_first_count"] = all_agent_values_df["agent_config"].str.split("-").str[1].astype(int)

    all_agent_values_df_all_steps = pd.concat(agent_data_list_all_steps)
    all_agent_values_df_all_steps["goal_first_count"] = all_agent_values_df_all_steps["agent_config"].str.split("-").str[1].astype(int)
    return all_agent_values_df, all_agent_values_df_all_steps


In [6]:
final_agent_values, all_agent_values_df_all_steps = create_agent_level_analysis(experiment_name="monitoring_vs_goal_based_analysis", experiment_date="20251108")
#agent_mean_df
final_agent_values


Creating Agent-Level Analysis for: monitoring_vs_goal_based_analysis, date: 20251108
Figures will be saved to: /Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results/monitoring_vs_goal_based_analysis_20251108/figures
Found experiment-specific configs in: /Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results/monitoring_vs_goal_based_analysis_20251108/configs
Found matching configs in subdirectory: /Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results/monitoring_vs_goal_based_analysis_20251108/configs (110 files)


/var/folders/qz/c3nhpt0n4_7g6vm6v12rwpxm0000gn/T/ipykernel_96207/2371806449.py:161: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  agent_df["seed"] = seed
/var/folders/qz/c3nhpt0n4_7g6vm6v12rwpxm0000gn/T/ipykernel_96207/2371806449.py:162: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  agent_df["agent_config"] = agent_config
/var/folders/qz/c3nhpt0n4_7g6vm6v12rwpxm0000gn/T/ipykernel_96207/2371806449.py:161: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .l

,Step,AgentID,Agent Persona,Accomplished Goals,Remaining Goals,Resource Conflicts,Counter Conflict Goal Accomplishments,Finished Step,Longest Idle Time,Total Idle Time,...,Number of Consents as G,Number of Consents as R Violated,Number of Consents as R Fulfilled,Number of Consents as R Unrealized,Number of Consents as G Violated,Number of Consents as G Fulfilled,Number of Consents as G Unrealized,seed,agent_config,goal_first_count
47000,47,1,GoalFirstAgent,3,0,13,10,24.0,21,21,...,31,5,4,0,10,15,6,13,0-500-0-500,500
47001,47,2,GoalFirstAgent,3,0,9,1,15.0,12,12,...,6,2,2,0,2,4,0,13,0-500-0-500,500
47002,47,3,GoalFirstAgent,3,0,11,3,22.0,11,19,...,4,6,4,0,2,1,1,13,0-500-0-500,500
47003,47,4,GoalFirstAgent,3,0,1,1,20.0,17,17,...,17,6,4,0,3,13,0,13,0-500-0-500,500
47004,47,5,GoalFirstAgent,3,0,12,1,25.0,12,22,...,21,4,6,0,4,13,3,13,0-500-0-500,500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11995,11,996,GoalFirstAgent,1,2,0,0,NaN,10,10,...,1,0,3,0,1,0,0,42,0-1000-0-0,1000
11996,11,997,GoalFirstAgent,1,2,7,0,NaN,10,10,...,2,3,2,0,2,0,0,42,0-1000-0-0,1000
11997,11,998,GoalFirstAgent,0,3,7,0,NaN,11,11,...,4,3,0,0,2,2,0,42,0-1000-0-0,1000
11998,11,999,GoalFirstAgent,0,3,7,0,NaN,11,11,...,4,6,0,0,2,2,0,42,0-1000-0-0,1000


In [7]:
all_agent_values_df_all_steps[["seed", "agent_config", "goal_first_count", "Step",  "AgentID", "Agent Persona", "Accomplished Goals"]]

,seed,agent_config,goal_first_count,Step,AgentID,Agent Persona,Accomplished Goals
0,13,0-500-0-500,500,0,1,GoalFirstAgent,0
1,13,0-500-0-500,500,0,2,GoalFirstAgent,0
2,13,0-500-0-500,500,0,3,GoalFirstAgent,0
3,13,0-500-0-500,500,0,4,GoalFirstAgent,0
4,13,0-500-0-500,500,0,5,GoalFirstAgent,0
...,...,...,...,...,...,...,...
11995,42,0-1000-0-0,1000,11,996,GoalFirstAgent,1
11996,42,0-1000-0-0,1000,11,997,GoalFirstAgent,1
11997,42,0-1000-0-0,1000,11,998,GoalFirstAgent,0
11998,42,0-1000-0-0,1000,11,999,GoalFirstAgent,0


In [8]:
df_sorted = final_agent_values.copy()

# Compute agent-level ratios
df_sorted = df_sorted[df_sorted["Number of Consents as R"] > 0]
df_sorted["consent_violation_ratio"] = (
    df_sorted["Number of Consents as R Violated"] /
    df_sorted["Number of Consents as R"]
)

df_sorted["consent_fulfillment_ratio"] = (
    df_sorted["Number of Consents as R Fulfilled"] /
    df_sorted["Number of Consents as R"]
)

df_sorted["tot_idle_time_normalized"] = df_sorted["Total Idle Time"] / df_sorted["Step"]

# ---- AGGREGATE per simulation run ----
run_level_df = (
    df_sorted.groupby(["goal_first_count", "seed", "Agent Persona"])[
        ["consent_violation_ratio", "consent_fulfillment_ratio", "Accomplished Goals", "tot_idle_time_normalized"]
    ]
    .mean()
    .reset_index()
)

df_va = run_level_df[run_level_df["Agent Persona"] == "MonitoringAgent"]
df_ta = run_level_df[run_level_df["Agent Persona"] == "GoalFirstAgent"]

# Create separate dataframes for "Accomplished Goals" and "tot_idle_time_normalized" 
# that include ALL agents (not just those with consents)
df_all_agents = final_agent_values.copy()
df_all_agents["tot_idle_time_normalized"] = df_all_agents["Total Idle Time"] / df_all_agents["Step"]

run_level_df_all_agents = (
    df_all_agents.groupby(["goal_first_count", "seed", "Agent Persona"])[
        ["Accomplished Goals", "tot_idle_time_normalized"]
    ]
    .mean()
    .reset_index()
)

df_va_full = run_level_df_all_agents[run_level_df_all_agents["Agent Persona"] == "MonitoringAgent"]
df_ta_full = run_level_df_all_agents[run_level_df_all_agents["Agent Persona"] == "GoalFirstAgent"]


## 04 Consent Violation Ratio, 1-WAY ANOVA TESTS

In [9]:
groups_va = [
    df_va[df_va["goal_first_count"] == r]["consent_violation_ratio"]
    for r in sorted(df_va["goal_first_count"].unique())
]

# One-way ANOVA for MonitoringAgent (VA)
F_va, p_va = stats.f_oneway(*groups_va)

# Effect size (eta-squared) for VA
k_va = len(groups_va)                       # number of groups
ns_va = [len(g) for g in groups_va]
N_va = sum(ns_va)                           # total sample size
df_between_va = k_va - 1
df_within_va = N_va - k_va
eta_sq_va = (F_va * df_between_va) / (F_va * df_between_va + df_within_va)

# Get min / max values of the averages
max_consent_violation_ratio_va = df_va.groupby("goal_first_count")["consent_violation_ratio"].mean().max()
min_consent_violation_ratio_va = df_va.groupby("goal_first_count")["consent_violation_ratio"].mean().min()

print(f"MonitoringAgent (VA) - F: {F_va:.4f}, p: {p_va:.3e}, eta^2: {eta_sq_va:.4f}")
print(f"  Max: {max_consent_violation_ratio_va:.4f}, Min: {min_consent_violation_ratio_va:.4f}")

groups_ta = [
    df_ta[df_ta["goal_first_count"] == r]["consent_violation_ratio"]
    for r in sorted(df_ta["goal_first_count"].unique())
]

# One-way ANOVA for GoalFirstAgent (TA)
F_ta, p_ta = stats.f_oneway(*groups_ta)

# Effect size (eta-squared) for TA
k_ta = len(groups_ta)                       # number of groups
ns_ta = [len(g) for g in groups_ta]
N_ta = sum(ns_ta)                           # total sample size
df_between_ta = k_ta - 1
df_within_ta = N_ta - k_ta
eta_sq_ta = (F_ta * df_between_ta) / (F_ta * df_between_ta + df_within_ta)

# Get min / max values of the averages
max_consent_violation_ratio_ta = df_ta.groupby("goal_first_count")["consent_violation_ratio"].mean().max()
min_consent_violation_ratio_ta = df_ta.groupby("goal_first_count")["consent_violation_ratio"].mean().min()

print(f"GoalFirstAgent (TA) - F: {F_ta:.4f}, p: {p_ta:.3e}, eta^2: {eta_sq_ta:.4f}")
print(f"  Max: {max_consent_violation_ratio_ta:.4f}, Min: {min_consent_violation_ratio_ta:.4f}")

MonitoringAgent (VA) - F: 363.0266, p: 1.061e-66, eta^2: 0.9732
  Max: 0.5552, Min: 0.1656
GoalFirstAgent (TA) - F: 228.4249, p: 5.623e-58, eta^2: 0.9581
  Max: 0.7839, Min: 0.3559


## 05 Consent Fulfilment Ratio, 1-WAY ANOVA TESTS

In [10]:
groups_va = [
    df_va[df_va["goal_first_count"] == r]["consent_fulfillment_ratio"]
    for r in sorted(df_va["goal_first_count"].unique())
]

# One-way ANOVA for MonitoringAgent (VA) - Fulfilment Ratio
F_va, p_va = stats.f_oneway(*groups_va)

# Effect size (eta-squared) for VA
k_va = len(groups_va)                       # number of groups
ns_va = [len(g) for g in groups_va]
N_va = sum(ns_va)                           # total sample size
df_between_va = k_va - 1
df_within_va = N_va - k_va
eta_sq_va = (F_va * df_between_va) / (F_va * df_between_va + df_within_va)

# Get min / max values of the averages
max_consent_fulfillment_ratio_va = df_va.groupby("goal_first_count")["consent_fulfillment_ratio"].mean().max()
min_consent_fulfillment_ratio_va = df_va.groupby("goal_first_count")["consent_fulfillment_ratio"].mean().min()

print(f"MonitoringAgent (VA) - F: {F_va:.4f}, p: {p_va:.3e}, eta^2: {eta_sq_va:.4f}")
print(f"  Max: {max_consent_fulfillment_ratio_va:.4f}, Min: {min_consent_fulfillment_ratio_va:.4f}")

groups_ta = [
    df_ta[df_ta["goal_first_count"] == r]["consent_fulfillment_ratio"]
    for r in sorted(df_ta["goal_first_count"].unique())
]

# One-way ANOVA for GoalFirstAgent (TA) - Fulfilment Ratio
F_ta, p_ta = stats.f_oneway(*groups_ta)

# Effect size (eta-squared) for TA
k_ta = len(groups_ta)                       # number of groups
ns_ta = [len(g) for g in groups_ta]
N_ta = sum(ns_ta)                           # total sample size
df_between_ta = k_ta - 1
df_within_ta = N_ta - k_ta
eta_sq_ta = (F_ta * df_between_ta) / (F_ta * df_between_ta + df_within_ta)

# Get min / max values of the averages
max_consent_fulfillment_ratio_ta = df_ta.groupby("goal_first_count")["consent_fulfillment_ratio"].mean().max()
min_consent_fulfillment_ratio_ta = df_ta.groupby("goal_first_count")["consent_fulfillment_ratio"].mean().min()

print(f"GoalFirstAgent (TA) - F: {F_ta:.4f}, p: {p_ta:.3e}, eta^2: {eta_sq_ta:.4f}")
print(f"  Max: {max_consent_fulfillment_ratio_ta:.4f}, Min: {min_consent_fulfillment_ratio_ta:.4f}")

MonitoringAgent (VA) - F: 612.6048, p: 1.074e-76, eta^2: 0.9839
  Max: 0.7105, Min: 0.1767
GoalFirstAgent (TA) - F: 482.4749, p: 4.043e-72, eta^2: 0.9797
  Max: 0.6441, Min: 0.2147


## 04: Consent Violation Ratio Mann-Whitney Test


In [11]:
import pandas as pd
from scipy.stats import mannwhitneyu

# ----------------------------------------------------------------------
# 1. Prepare data properly: compute ratios + aggregate per run
# ----------------------------------------------------------------------

df_sorted = final_agent_values.copy()

# Keep only agents with at least 1 consent as R
df_sorted = df_sorted[df_sorted["Number of Consents as R"] > 0]

# Compute ratios
df_sorted["consent_violation_ratio"] = (
    df_sorted["Number of Consents as R Violated"] /
    df_sorted["Number of Consents as R"]
)

df_sorted["consent_fulfillment_ratio"] = (
    df_sorted["Number of Consents as R Fulfilled"] /
    df_sorted["Number of Consents as R"]
)

run_level_df = df_sorted.groupby(["goal_first_count", "seed", "Agent Persona"])[[
    "Number of Consents as R",
    "Number of Consents as R Violated",
    "Number of Consents as R Fulfilled"
]].sum().reset_index()

# Aggregate so that each (ratio × seed × persona) is one data point
run_level_df["consent_violation_ratio"] = run_level_df["Number of Consents as R Violated"] / run_level_df["Number of Consents as R"]
run_level_df["consent_fulfillment_ratio"] = run_level_df["Number of Consents as R Fulfilled"] / run_level_df["Number of Consents as R"]


# Split by persona
df_va = run_level_df[run_level_df["Agent Persona"] == "MonitoringAgent"]
df_ta = run_level_df[run_level_df["Agent Persona"] == "GoalFirstAgent"]

# ----------------------------------------------------------------------
# 2. Compare violation ratios between personas
# ----------------------------------------------------------------------

# Means per condition (align conditions across personas)
monitoring_means = df_va.groupby("goal_first_count")["consent_violation_ratio"].mean()
goal_first_means = df_ta.groupby("goal_first_count")["consent_violation_ratio"].mean()

# Use union of all observed goal_first_count values and fill missing persona means with 0
all_counts = sorted(set(monitoring_means.index).union(set(goal_first_means.index)))
monitoring_aligned = monitoring_means.reindex(all_counts, fill_value=0)
goal_first_aligned = goal_first_means.reindex(all_counts, fill_value=0)

comparison_df = pd.DataFrame({
    "goal_first_count": all_counts,
    "MonitoringAgent_mean": monitoring_aligned.values,
    "GoalFirstAgent_mean": goal_first_aligned.values
})
comparison_df["Difference"] = (
    comparison_df["MonitoringAgent_mean"] -
    comparison_df["GoalFirstAgent_mean"]
)
comparison_df["Monitoring_higher"] = comparison_df["Difference"] > 0

print("\nViolation Ratio Comparison by Persona:")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)

# Count conditions
monitoring_higher_count = comparison_df["Monitoring_higher"].sum()
goal_first_higher_count = len(comparison_df) - monitoring_higher_count

print(f"\nMonitoringAgent higher in {monitoring_higher_count}/"
      f"{len(comparison_df)} conditions")
print(f"GoalFirstAgent higher in {goal_first_higher_count}/"
      f"{len(comparison_df)} conditions")

# Overall means using run-level data
overall_va = df_va["consent_violation_ratio"].mean()
overall_ta = df_ta["consent_violation_ratio"].mean()

print("\nOverall mean violation ratios:")
print(f"  MonitoringAgent: {overall_va:.4f}")
print(f"  GoalFirstAgent:  {overall_ta:.4f}")
print(f"  Difference:      {overall_va - overall_ta:.4f}")

# Mann–Whitney U tests
print("\nMann–Whitney U Test (Violation Ratios):")
cons_va = df_va["consent_violation_ratio"]
cons_ta = df_ta["consent_violation_ratio"]

u_twosided, p_twosided = mannwhitneyu(cons_va, cons_ta, alternative="two-sided")
u_va_greater, p_va_greater = mannwhitneyu(cons_va, cons_ta, alternative="greater")
u_ta_greater, p_ta_greater = mannwhitneyu(cons_ta, cons_va, alternative="greater")

print(f"  Two-sided:           p = {p_twosided:.3e}")
print(f"  VA > TA (one-sided): p = {p_va_greater:.3e}")
print(f"  TA > VA (one-sided): p = {p_ta_greater:.3e}")

# Rank-biserial correlation (effect size) using two-sided U
n_va = len(cons_va)
n_ta = len(cons_ta)
if n_va > 0 and n_ta > 0:
    r_rb_abs = 1 - (2 * u_twosided) / (n_va * n_ta)
    # Give direction based on overall means
    direction = 1 if overall_va > overall_ta else -1
    r_rb = direction * abs(r_rb_abs)
    print(f"  Rank-biserial r:    r_rb = {r_rb:.4f} (|r| = {abs(r_rb_abs):.4f})")

if p_twosided < 0.05:
    if overall_va > overall_ta:
        print("  → SIGNIFICANT: MonitoringAgent violate more.")
    else:
        print("  → SIGNIFICANT: GoalFirstAgent violate more.")
else:
    print("  → NO significant difference.")


# ----------------------------------------------------------------------
# 3. (Optional) Repeat EXACT same structure for fulfilment ratios
#    Just replace "consent_violation_ratio" with "consent_fulfillment_ratio"
# ----------------------------------------------------------------------



Violation Ratio Comparison by Persona:
 goal_first_count  MonitoringAgent_mean  GoalFirstAgent_mean  Difference  Monitoring_higher
                0              0.186037             0.000000    0.186037               True
              100              0.201979             0.352517   -0.150538              False
              200              0.217707             0.369755   -0.152047              False
              300              0.235656             0.381902   -0.146246              False
              400              0.267332             0.401464   -0.134132              False
              500              0.303957             0.417462   -0.113505              False
              600              0.355312             0.443347   -0.088035              False
              700              0.447740             0.494670   -0.046931              False
              800              0.561723             0.569804   -0.008081              False
              900              0.629737 

## 05 Consent Fulfilment Ratio Mann-Whitney

In [12]:
# ----------------------------------------------------------------------
# 1. Prepare data properly: compute ratios + aggregate per run
# ----------------------------------------------------------------------


# Split into two persona sets
df_va = run_level_df[run_level_df["Agent Persona"] == "MonitoringAgent"]
df_ta = run_level_df[run_level_df["Agent Persona"] == "GoalFirstAgent"]

# ----------------------------------------------------------------------
# 2. Compare fulfilment ratios between personas
# ----------------------------------------------------------------------

# Mean per condition (align conditions across personas)
monitoring_means = df_va.groupby("goal_first_count")["consent_fulfillment_ratio"].mean()
goal_first_means = df_ta.groupby("goal_first_count")["consent_fulfillment_ratio"].mean()

# Use union of all observed goal_first_count values and fill missing persona means with 0
all_counts = sorted(set(monitoring_means.index).union(set(goal_first_means.index)))
monitoring_aligned = monitoring_means.reindex(all_counts, fill_value=0)
goal_first_aligned = goal_first_means.reindex(all_counts, fill_value=0)

comparison_df = pd.DataFrame({
    "goal_first_count": all_counts,
    "MonitoringAgent_mean": monitoring_aligned.values,
    "GoalFirstAgent_mean": goal_first_aligned.values
})
comparison_df["Difference"] = (
    comparison_df["MonitoringAgent_mean"] -
    comparison_df["GoalFirstAgent_mean"]
)
comparison_df["Monitoring_higher"] = comparison_df["Difference"] > 0

print("\nFulfillment Ratio Comparison by Persona:")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)

monitoring_higher_count = comparison_df["Monitoring_higher"].sum()
goal_first_higher_count    = len(comparison_df) - monitoring_higher_count

print(f"\nMonitoringAgent higher in {monitoring_higher_count}/"
      f"{len(comparison_df)} conditions")
print(f"GoalFirstAgent   higher in {goal_first_higher_count}/"
      f"{len(comparison_df)} conditions")

# ----------------------------------------------------------------------
# 3. Compare overall means
# ----------------------------------------------------------------------

overall_va = df_va["consent_fulfillment_ratio"].mean()
overall_ta = df_ta["consent_fulfillment_ratio"].mean()

print("\nOverall mean fulfilment ratios:")
print(f"  MonitoringAgent: {overall_va:.4f}")
print(f"  GoalFirstAgent:  {overall_ta:.4f}")
print(f"  Difference:      {overall_va - overall_ta:.4f}")

# ----------------------------------------------------------------------
# 4. Mann–Whitney U tests (non-parametric)
# ----------------------------------------------------------------------

print("\nMann–Whitney U Test (Fulfillment Ratios):")
cons_va = df_va["consent_fulfillment_ratio"]
cons_ta = df_ta["consent_fulfillment_ratio"]

u_twosided,      p_twosided      = mannwhitneyu(cons_va, cons_ta, alternative="two-sided")
u_va_greater,    p_va_greater    = mannwhitneyu(cons_va, cons_ta, alternative="greater")
u_ta_greater,    p_ta_greater    = mannwhitneyu(cons_ta, cons_va, alternative="greater")

print(f"  Two-sided:           p = {p_twosided:.3e}")
print(f"  VA > TA (one-sided): p = {p_va_greater:.3e}")
print(f"  TA > VA (one-sided): p = {p_ta_greater:.3e}")

# Rank-biserial correlation (effect size) using two-sided U
n_va = len(cons_va)
n_ta = len(cons_ta)
if n_va > 0 and n_ta > 0:
    r_rb_abs = 1 - (2 * u_twosided) / (n_va * n_ta)
    # Give direction based on overall means
    direction = 1 if overall_va > overall_ta else -1
    r_rb = direction * abs(r_rb_abs)
    print(f"  Rank-biserial r:    r_rb = {r_rb:.4f} (|r| = {abs(r_rb_abs):.4f})")

if p_twosided < 0.05:
    if overall_va > overall_ta:
        print("  → SIGNIFICANT: MonitoringAgent fulfils more.")
    else:
        print("  → SIGNIFICANT: GoalFirstAgent fulfils more.")
else:
    print("  → NO significant difference.")

# ----------------------------------------------------------------------
# 5. Mann–Whitney U test for goal_first_count >= 600 (after intersection)
# ----------------------------------------------------------------------

print("\nMann–Whitney U Test (Fulfillment Ratios, goal_first_count >= 600):")
# Filter to only goal_first_count >= 600 where both personas exist
# Find common goal_first_count values >= 600 where both personas have data
va_counts_high = set(df_va[df_va["goal_first_count"] >= 600]["goal_first_count"].unique())
ta_counts_high = set(df_ta[df_ta["goal_first_count"] >= 600]["goal_first_count"].unique())
common_counts_high = sorted(va_counts_high.intersection(ta_counts_high))

df_va_high = df_va[df_va["goal_first_count"].isin(common_counts_high)]
df_ta_high = df_ta[df_ta["goal_first_count"].isin(common_counts_high)]

if len(df_va_high) > 0 and len(df_ta_high) > 0:
    cons_va_high = df_va_high["consent_fulfillment_ratio"]
    cons_ta_high = df_ta_high["consent_fulfillment_ratio"]
    
    u_twosided_high, p_twosided_high = mannwhitneyu(cons_va_high, cons_ta_high, alternative="two-sided")
    u_va_greater_high, p_va_greater_high = mannwhitneyu(cons_va_high, cons_ta_high, alternative="greater")
    u_ta_greater_high, p_ta_greater_high = mannwhitneyu(cons_ta_high, cons_va_high, alternative="greater")
    
    overall_va_high = cons_va_high.mean()
    overall_ta_high = cons_ta_high.mean()
    
    print(f"  Two-sided:           p = {p_twosided_high:.3e}")
    print(f"  VA > TA (one-sided): p = {p_va_greater_high:.3e}")
    print(f"  TA > VA (one-sided): p = {p_ta_greater_high:.3e}")
    print(f"  MonitoringAgent mean: {overall_va_high:.4f}")
    print(f"  GoalFirstAgent mean:  {overall_ta_high:.4f}")
    print(f"  Difference:           {overall_va_high - overall_ta_high:.4f}")
    
    # Rank-biserial correlation (effect size) using two-sided U
    n_va_high = len(cons_va_high)
    n_ta_high = len(cons_ta_high)
    if n_va_high > 0 and n_ta_high > 0:
        r_rb_abs_high = 1 - (2 * u_twosided_high) / (n_va_high * n_ta_high)
        # Give direction based on overall means
        direction_high = 1 if overall_va_high > overall_ta_high else -1
        r_rb_high = direction_high * abs(r_rb_abs_high)
        print(f"  Rank-biserial r:    r_rb = {r_rb_high:.4f} (|r| = {abs(r_rb_abs_high):.4f})")
    
    if p_twosided_high < 0.05:
        if overall_va_high > overall_ta_high:
            print("  → SIGNIFICANT: MonitoringAgent fulfils more (goal_first_count >= 600).")
        else:
            print("  → SIGNIFICANT: GoalFirstAgent fulfils more (goal_first_count >= 600).")
    else:
        print("  → NO significant difference (goal_first_count >= 600).")
else:
    print("  → Insufficient data for goal_first_count >= 600 comparison.")



Fulfillment Ratio Comparison by Persona:
 goal_first_count  MonitoringAgent_mean  GoalFirstAgent_mean  Difference  Monitoring_higher
                0              0.679991             0.000000    0.679991               True
              100              0.657202             0.647483    0.009719               True
              200              0.634475             0.630245    0.004229               True
              300              0.608455             0.618098   -0.009643              False
              400              0.568410             0.598536   -0.030127              False
              500              0.516623             0.582514   -0.065890              False
              600              0.446617             0.556530   -0.109913              False
              700              0.323822             0.505236   -0.181414              False
              800              0.190713             0.429921   -0.239208              False
              900              0.10860

## 05: Consent Fulfilment Ratio Mann-Whitney Test > 600


In [13]:
# ----------------------------------------------------------------------
# 1. Prepare data properly: compute ratios + aggregate per run
# ----------------------------------------------------------------------


# Split into two persona sets
df_va = run_level_df[run_level_df["Agent Persona"] == "MonitoringAgent"]
df_ta = run_level_df[run_level_df["Agent Persona"] == "GoalFirstAgent"]

# ----------------------------------------------------------------------
# 2. Compare fulfilment ratios between personas
# ----------------------------------------------------------------------

# Mean per condition (align conditions across personas)
monitoring_means = df_va.groupby("goal_first_count")["consent_fulfillment_ratio"].mean()
goal_first_means = df_ta.groupby("goal_first_count")["consent_fulfillment_ratio"].mean()

# Use union of all observed goal_first_count values and fill missing persona means with 0
all_counts = sorted(set(monitoring_means.index).union(set(goal_first_means.index)))
monitoring_aligned = monitoring_means.reindex(all_counts, fill_value=0)
goal_first_aligned = goal_first_means.reindex(all_counts, fill_value=0)

comparison_df = pd.DataFrame({
    "goal_first_count": all_counts,
    "MonitoringAgent_mean": monitoring_aligned.values,
    "GoalFirstAgent_mean": goal_first_aligned.values
})
comparison_df["Difference"] = (
    comparison_df["MonitoringAgent_mean"] -
    comparison_df["GoalFirstAgent_mean"]
)
comparison_df["Monitoring_higher"] = comparison_df["Difference"] > 0

print("\nFulfillment Ratio Comparison by Persona:")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)

monitoring_higher_count = comparison_df["Monitoring_higher"].sum()
goal_first_higher_count    = len(comparison_df) - monitoring_higher_count

print(f"\nMonitoringAgent higher in {monitoring_higher_count}/"
      f"{len(comparison_df)} conditions")
print(f"GoalFirstAgent   higher in {goal_first_higher_count}/"
      f"{len(comparison_df)} conditions")

# ----------------------------------------------------------------------
# 3. Compare overall means
# ----------------------------------------------------------------------

overall_va = df_va["consent_fulfillment_ratio"].mean()
overall_ta = df_ta["consent_fulfillment_ratio"].mean()

print("\nOverall mean fulfilment ratios:")
print(f"  MonitoringAgent: {overall_va:.4f}")
print(f"  GoalFirstAgent:  {overall_ta:.4f}")
print(f"  Difference:      {overall_va - overall_ta:.4f}")

# ----------------------------------------------------------------------
# 4. Mann–Whitney U tests (non-parametric)
# ----------------------------------------------------------------------

print("\nMann–Whitney U Test (Fulfillment Ratios):")
cons_va = df_va["consent_fulfillment_ratio"]
cons_ta = df_ta["consent_fulfillment_ratio"]

u_twosided,      p_twosided      = mannwhitneyu(cons_va, cons_ta, alternative="two-sided")
u_va_greater,    p_va_greater    = mannwhitneyu(cons_va, cons_ta, alternative="greater")
u_ta_greater,    p_ta_greater    = mannwhitneyu(cons_ta, cons_va, alternative="greater")

print(f"  Two-sided:           p = {p_twosided:.3e}")
print(f"  VA > TA (one-sided): p = {p_va_greater:.3e}")
print(f"  TA > VA (one-sided): p = {p_ta_greater:.3e}")

# Rank-biserial correlation (effect size) using two-sided U
n_va = len(cons_va)
n_ta = len(cons_ta)
if n_va > 0 and n_ta > 0:
    r_rb_abs = 1 - (2 * u_twosided) / (n_va * n_ta)
    # Give direction based on overall means
    direction = 1 if overall_va > overall_ta else -1
    r_rb = direction * abs(r_rb_abs)
    print(f"  Rank-biserial r:    r_rb = {r_rb:.4f} (|r| = {abs(r_rb_abs):.4f})")

if p_twosided < 0.05:
    if overall_va > overall_ta:
        print("  → SIGNIFICANT: MonitoringAgent fulfils more.")
    else:
        print("  → SIGNIFICANT: GoalFirstAgent fulfils more.")
else:
    print("  → NO significant difference.")

# ----------------------------------------------------------------------
# 5. Mann–Whitney U test for goal_first_count >= 600 (after intersection)
# ----------------------------------------------------------------------

print("\nMann–Whitney U Test (Fulfillment Ratios, goal_first_count >= 600):")
# Filter to only goal_first_count >= 600 where both personas exist
# Find common goal_first_count values >= 600 where both personas have data
va_counts_high = set(df_va[df_va["goal_first_count"] >= 600]["goal_first_count"].unique())
ta_counts_high = set(df_ta[df_ta["goal_first_count"] >= 600]["goal_first_count"].unique())
common_counts_high = sorted(va_counts_high.intersection(ta_counts_high))

df_va_high = df_va[df_va["goal_first_count"].isin(common_counts_high)]
df_ta_high = df_ta[df_ta["goal_first_count"].isin(common_counts_high)]

if len(df_va_high) > 0 and len(df_ta_high) > 0:
    cons_va_high = df_va_high["consent_fulfillment_ratio"]
    cons_ta_high = df_ta_high["consent_fulfillment_ratio"]
    
    u_twosided_high, p_twosided_high = mannwhitneyu(cons_va_high, cons_ta_high, alternative="two-sided")
    u_va_greater_high, p_va_greater_high = mannwhitneyu(cons_va_high, cons_ta_high, alternative="greater")
    u_ta_greater_high, p_ta_greater_high = mannwhitneyu(cons_ta_high, cons_va_high, alternative="greater")
    
    overall_va_high = cons_va_high.mean()
    overall_ta_high = cons_ta_high.mean()
    
    print(f"  Two-sided:           p = {p_twosided_high:.3e}")
    print(f"  VA > TA (one-sided): p = {p_va_greater_high:.3e}")
    print(f"  TA > VA (one-sided): p = {p_ta_greater_high:.3e}")
    print(f"  MonitoringAgent mean: {overall_va_high:.4f}")
    print(f"  GoalFirstAgent mean:  {overall_ta_high:.4f}")
    print(f"  Difference:           {overall_va_high - overall_ta_high:.4f}")
    
    # Rank-biserial correlation (effect size) using two-sided U
    n_va_high = len(cons_va_high)
    n_ta_high = len(cons_ta_high)
    if n_va_high > 0 and n_ta_high > 0:
        r_rb_abs_high = 1 - (2 * u_twosided_high) / (n_va_high * n_ta_high)
        # Give direction based on overall means
        direction_high = 1 if overall_va_high > overall_ta_high else -1
        r_rb_high = direction_high * abs(r_rb_abs_high)
        print(f"  Rank-biserial r:    r_rb = {r_rb_high:.4f} (|r| = {abs(r_rb_abs_high):.4f})")
    
    if p_twosided_high < 0.05:
        if overall_va_high > overall_ta_high:
            print("  → SIGNIFICANT: MonitoringAgent fulfils more (goal_first_count >= 600).")
        else:
            print("  → SIGNIFICANT: GoalFirstAgent fulfils more (goal_first_count >= 600).")
    else:
        print("  → NO significant difference (goal_first_count >= 600).")
else:
    print("  → Insufficient data for goal_first_count >= 600 comparison.")



Fulfillment Ratio Comparison by Persona:
 goal_first_count  MonitoringAgent_mean  GoalFirstAgent_mean  Difference  Monitoring_higher
                0              0.679991             0.000000    0.679991               True
              100              0.657202             0.647483    0.009719               True
              200              0.634475             0.630245    0.004229               True
              300              0.608455             0.618098   -0.009643              False
              400              0.568410             0.598536   -0.030127              False
              500              0.516623             0.582514   -0.065890              False
              600              0.446617             0.556530   -0.109913              False
              700              0.323822             0.505236   -0.181414              False
              800              0.190713             0.429921   -0.239208              False
              900              0.10860

## 06: Accomplished Goals 1-Way ANOVA Test


In [14]:
# Use df_va_full and df_ta_full from cell 12 which includes "Accomplished Goals"
# (saved before they were overwritten in cells 18 and 20)
# These use FULL data (all agents, not filtered by consent count > 0)
groups_va = [
    df_va_full[df_va_full["goal_first_count"] == r]["Accomplished Goals"]
    for r in sorted(df_va_full["goal_first_count"].unique())
]

# One-way ANOVA for MonitoringAgent (VA) - Accomplished Goals
F_va, p_va = stats.f_oneway(*groups_va)

# Effect size (eta-squared) for VA
k_va = len(groups_va)                       # number of groups
ns_va = [len(g) for g in groups_va]
N_va = sum(ns_va)                           # total sample size
df_between_va = k_va - 1
df_within_va = N_va - k_va
eta_sq_va = (F_va * df_between_va) / (F_va * df_between_va + df_within_va)

# Get min / max values of the averages
max_accomplished_goals_va = df_va_full.groupby("goal_first_count")["Accomplished Goals"].mean().max()
min_accomplished_goals_va = df_va_full.groupby("goal_first_count")["Accomplished Goals"].mean().min()

print(f"MonitoringAgent (VA) - F: {F_va:.4f}, p: {p_va:.3e}, eta^2: {eta_sq_va:.4f}")
print(f"  Max: {max_accomplished_goals_va:.4f}, Min: {min_accomplished_goals_va:.4f}")

groups_ta = [
    df_ta_full[df_ta_full["goal_first_count"] == r]["Accomplished Goals"]
    for r in sorted(df_ta_full["goal_first_count"].unique())
]

# One-way ANOVA for GoalFirstAgent (TA) - Accomplished Goals
F_ta, p_ta = stats.f_oneway(*groups_ta)

# Effect size (eta-squared) for TA
k_ta = len(groups_ta)                       # number of groups
ns_ta = [len(g) for g in groups_ta]
N_ta = sum(ns_ta)                           # total sample size
df_between_ta = k_ta - 1
df_within_ta = N_ta - k_ta
eta_sq_ta = (F_ta * df_between_ta) / (F_ta * df_between_ta + df_within_ta)

# Get min / max values of the averages
max_accomplished_goals_ta = df_ta_full.groupby("goal_first_count")["Accomplished Goals"].mean().max()
min_accomplished_goals_ta = df_ta_full.groupby("goal_first_count")["Accomplished Goals"].mean().min()

print(f"GoalFirstAgent (TA) - F: {F_ta:.4f}, p: {p_ta:.3e}, eta^2: {eta_sq_ta:.4f}")
print(f"  Max: {max_accomplished_goals_ta:.4f}, Min: {min_accomplished_goals_ta:.4f}")

MonitoringAgent (VA) - F: 358.7367, p: 1.784e-66, eta^2: 0.9729
  Max: 3.0000, Min: 0.8370
GoalFirstAgent (TA) - F: 608.0205, p: 1.496e-76, eta^2: 0.9838
  Max: 2.9860, Min: 0.5121


## 08: Total Idle Time per Agent Normalized by Steps

In [15]:
# Use df_va_full and df_ta_full from cell 12 which includes "tot_idle_time_normalized"
# (saved before they were overwritten in cells 18 and 20)
# These use FULL data (all agents, not filtered by consent count > 0)
groups_va = [
    df_va_full[df_va_full["goal_first_count"] == r]["tot_idle_time_normalized"]
    for r in sorted(df_va_full["goal_first_count"].unique())
]

# One-way ANOVA for MonitoringAgent (VA) - Total Idle Time Normalized
F_va, p_va = stats.f_oneway(*groups_va)

# Effect size (eta-squared) for VA
k_va = len(groups_va)                       # number of groups
ns_va = [len(g) for g in groups_va]
N_va = sum(ns_va)                           # total sample size
df_between_va = k_va - 1
df_within_va = N_va - k_va
eta_sq_va = (F_va * df_between_va) / (F_va * df_between_va + df_within_va)

# Get min / max values of the averages
max_tot_idle_time_normalized_va = df_va_full.groupby("goal_first_count")["tot_idle_time_normalized"].mean().max()
min_tot_idle_time_normalized_va = df_va_full.groupby("goal_first_count")["tot_idle_time_normalized"].mean().min()

print(f"MonitoringAgent (VA) - F: {F_va:.4f}, p: {p_va:.3e}, eta^2: {eta_sq_va:.4f}")
print(f"  Max: {max_tot_idle_time_normalized_va:.4f}, Min: {min_tot_idle_time_normalized_va:.4f}")

groups_ta = [
    df_ta_full[df_ta_full["goal_first_count"] == r]["tot_idle_time_normalized"]
    for r in sorted(df_ta_full["goal_first_count"].unique())
]

# One-way ANOVA for GoalFirstAgent (TA) - Total Idle Time Normalized
F_ta, p_ta = stats.f_oneway(*groups_ta)

# Effect size (eta-squared) for TA
k_ta = len(groups_ta)                       # number of groups
ns_ta = [len(g) for g in groups_ta]
N_ta = sum(ns_ta)                           # total sample size
df_between_ta = k_ta - 1
df_within_ta = N_ta - k_ta
eta_sq_ta = (F_ta * df_between_ta) / (F_ta * df_between_ta + df_within_ta)

# Get min / max values of the averages
max_tot_idle_time_normalized_ta = df_ta_full.groupby("goal_first_count")["tot_idle_time_normalized"].mean().max()
min_tot_idle_time_normalized_ta = df_ta_full.groupby("goal_first_count")["tot_idle_time_normalized"].mean().min()

print(f"GoalFirstAgent (TA) - F: {F_ta:.4f}, p: {p_ta:.3e}, eta^2: {eta_sq_ta:.4f}")
print(f"  Max: {max_tot_idle_time_normalized_ta:.4f}, Min: {min_tot_idle_time_normalized_ta:.4f}")

MonitoringAgent (VA) - F: 184.3425, p: 5.375e-54, eta^2: 0.9485
  Max: 0.9254, Min: 0.3987
GoalFirstAgent (TA) - F: 211.2267, p: 1.614e-56, eta^2: 0.9548
  Max: 0.9390, Min: 0.4420


## 08: Total Idle Time Normalized Mann-Whitney Test


In [16]:
# ----------------------------------------------------------------------
# 1. Use df_va_full and df_ta_full from cell 12 which includes "tot_idle_time_normalized"
#    These include ALL agents (not just those with consents)
# ----------------------------------------------------------------------

# ----------------------------------------------------------------------
# 2. Compare tot_idle_time_normalized between personas
# ----------------------------------------------------------------------

# Means per condition (align conditions across personas)
monitoring_means = df_va_full.groupby("goal_first_count")["tot_idle_time_normalized"].mean()
goal_first_means = df_ta_full.groupby("goal_first_count")["tot_idle_time_normalized"].mean()

# Use union of all observed goal_first_count values and fill missing persona means with 0
all_counts = sorted(set(monitoring_means.index).union(set(goal_first_means.index)))
monitoring_aligned = monitoring_means.reindex(all_counts, fill_value=0)
goal_first_aligned = goal_first_means.reindex(all_counts, fill_value=0)

comparison_df = pd.DataFrame({
    "goal_first_count": all_counts,
    "MonitoringAgent_mean": monitoring_aligned.values,
    "GoalFirstAgent_mean": goal_first_aligned.values
})
comparison_df["Difference"] = (
    comparison_df["MonitoringAgent_mean"] -
    comparison_df["GoalFirstAgent_mean"]
)
comparison_df["Monitoring_higher"] = comparison_df["Difference"] > 0

print("\nTotal Idle Time Normalized Comparison by Persona:")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)

# Count conditions
monitoring_higher_count = comparison_df["Monitoring_higher"].sum()
goal_first_higher_count = len(comparison_df) - monitoring_higher_count

print(f"\nMonitoringAgent higher in {monitoring_higher_count}/"
      f"{len(comparison_df)} conditions")
print(f"GoalFirstAgent higher in {goal_first_higher_count}/"
      f"{len(comparison_df)} conditions")

# ----------------------------------------------------------------------
# 3. Compare overall means
# ----------------------------------------------------------------------

overall_va = df_va_full["tot_idle_time_normalized"].mean()
overall_ta = df_ta_full["tot_idle_time_normalized"].mean()

print("\nOverall mean tot_idle_time_normalized:")
print(f"  MonitoringAgent: {overall_va:.4f}")
print(f"  GoalFirstAgent:  {overall_ta:.4f}")
print(f"  Difference:      {overall_va - overall_ta:.4f}")

# ----------------------------------------------------------------------
# 4. Mann–Whitney U tests (non-parametric)
# ----------------------------------------------------------------------

print("\nMann–Whitney U Test (Total Idle Time Normalized):")
idle_va = df_va_full["tot_idle_time_normalized"]
idle_ta = df_ta_full["tot_idle_time_normalized"]

u_twosided, p_twosided = mannwhitneyu(idle_va, idle_ta, alternative="two-sided")
u_va_greater, p_va_greater = mannwhitneyu(idle_va, idle_ta, alternative="greater")
u_ta_greater, p_ta_greater = mannwhitneyu(idle_ta, idle_va, alternative="greater")

print(f"  Two-sided:           p = {p_twosided:.3e}")
print(f"  VA > TA (one-sided): p = {p_va_greater:.3e}")
print(f"  TA > VA (one-sided): p = {p_ta_greater:.3e}")

# Rank-biserial correlation (effect size) using two-sided U
n_va = len(idle_va)
n_ta = len(idle_ta)
if n_va > 0 and n_ta > 0:
    r_rb_abs = 1 - (2 * u_twosided) / (n_va * n_ta)
    # Give direction based on overall means
    direction = 1 if overall_va > overall_ta else -1
    r_rb = direction * abs(r_rb_abs)
    print(f"  Rank-biserial r:    r_rb = {r_rb:.4f} (|r| = {abs(r_rb_abs):.4f})")

if p_twosided < 0.05:
    if overall_va > overall_ta:
        print("  → SIGNIFICANT: MonitoringAgent has higher normalized idle time.")
    else:
        print("  → SIGNIFICANT: GoalFirstAgent has higher normalized idle time.")
else:
    print("  → NO significant difference.")



Total Idle Time Normalized Comparison by Persona:
 goal_first_count  MonitoringAgent_mean  GoalFirstAgent_mean  Difference  Monitoring_higher
                0              0.511969             0.000000    0.511969               True
              100              0.429095             0.490309   -0.061213              False
              200              0.429587             0.480910   -0.051324              False
              300              0.398716             0.441993   -0.043277              False
              400              0.424726             0.462107   -0.037381              False
              500              0.438216             0.471032   -0.032816              False
              600              0.497920             0.527691   -0.029770              False
              700              0.595498             0.621704   -0.026206              False
              800              0.745075             0.775113   -0.030037              False
              900            

## 10: Average Distinct Agents Interacted as R, 1-Way ANOVA Tests


In [17]:
# Create run-level dataframe for "Number of Distinct Agents Interacted as R"
# This includes ALL agents (not just those with consents)
df_distinct_r = final_agent_values.copy()

run_level_df_distinct_r = (
    df_distinct_r.groupby(["goal_first_count", "seed", "Agent Persona"])[
        ["Number of Distinct Agents Interacted as R"]
    ]
    .mean()
    .reset_index()
)

df_va_distinct_r = run_level_df_distinct_r[run_level_df_distinct_r["Agent Persona"] == "MonitoringAgent"]
df_ta_distinct_r = run_level_df_distinct_r[run_level_df_distinct_r["Agent Persona"] == "GoalFirstAgent"]

# 1-way ANOVA for MonitoringAgent (VA)
groups_va = [
    df_va_distinct_r[df_va_distinct_r["goal_first_count"] == r]["Number of Distinct Agents Interacted as R"]
    for r in sorted(df_va_distinct_r["goal_first_count"].unique())
]

F_va, p_va = stats.f_oneway(*groups_va)

# Effect size (eta-squared) for VA
k_va = len(groups_va)                       # number of groups
ns_va = [len(g) for g in groups_va]
N_va = sum(ns_va)                           # total sample size
df_between_va = k_va - 1
df_within_va = N_va - k_va
eta_sq_va = (F_va * df_between_va) / (F_va * df_between_va + df_within_va)

# Get min / max values of the averages
max_distinct_agents_r_va = df_va_distinct_r.groupby("goal_first_count")["Number of Distinct Agents Interacted as R"].mean().max()
min_distinct_agents_r_va = df_va_distinct_r.groupby("goal_first_count")["Number of Distinct Agents Interacted as R"].mean().min()

print(f"MonitoringAgent (VA) - F: {F_va:.4f}, p: {p_va:.3e}, eta^2: {eta_sq_va:.4f}")
print(f"  Max: {max_distinct_agents_r_va:.4f}, Min: {min_distinct_agents_r_va:.4f}")

# 1-way ANOVA for GoalFirstAgent (TA)
groups_ta = [
    df_ta_distinct_r[df_ta_distinct_r["goal_first_count"] == r]["Number of Distinct Agents Interacted as R"]
    for r in sorted(df_ta_distinct_r["goal_first_count"].unique())
]

F_ta, p_ta = stats.f_oneway(*groups_ta)

# Effect size (eta-squared) for TA
k_ta = len(groups_ta)                       # number of groups
ns_ta = [len(g) for g in groups_ta]
N_ta = sum(ns_ta)                           # total sample size
df_between_ta = k_ta - 1
df_within_ta = N_ta - k_ta
eta_sq_ta = (F_ta * df_between_ta) / (F_ta * df_between_ta + df_within_ta)

# Get min / max values of the averages
max_distinct_agents_r_ta = df_ta_distinct_r.groupby("goal_first_count")["Number of Distinct Agents Interacted as R"].mean().max()
min_distinct_agents_r_ta = df_ta_distinct_r.groupby("goal_first_count")["Number of Distinct Agents Interacted as R"].mean().min()

print(f"GoalFirstAgent (TA) - F: {F_ta:.4f}, p: {p_ta:.3e}, eta^2: {eta_sq_ta:.4f}")
print(f"  Max: {max_distinct_agents_r_ta:.4f}, Min: {min_distinct_agents_r_ta:.4f}")


MonitoringAgent (VA) - F: 17.2701, p: 3.262e-16, eta^2: 0.6333
  Max: 13.6195, Min: 8.5830
GoalFirstAgent (TA) - F: 710.0154, p: 1.561e-79, eta^2: 0.9861
  Max: 7.7975, Min: 3.4239


## 11: Average Distinct Agents Interacted as R, Mann-Whitney Test



In [21]:
# ----------------------------------------------------------------------
# 1. Use df_va_distinct_r and df_ta_distinct_r from cell 28 (ANOVA test)
#    These include ALL agents (not just those with consents)
# ----------------------------------------------------------------------

# ----------------------------------------------------------------------
# 2. Compare "Number of Distinct Agents Interacted as R" between personas
# ----------------------------------------------------------------------

# Means per condition (align conditions across personas)
monitoring_means = df_va_distinct_r.groupby("goal_first_count")["Number of Distinct Agents Interacted as R"].mean()
goal_first_means = df_ta_distinct_r.groupby("goal_first_count")["Number of Distinct Agents Interacted as R"].mean()

# Use union of all observed goal_first_count values and fill missing persona means with 0
all_counts = sorted(set(monitoring_means.index).union(set(goal_first_means.index)))
monitoring_aligned = monitoring_means.reindex(all_counts, fill_value=0)
goal_first_aligned = goal_first_means.reindex(all_counts, fill_value=0)

comparison_df = pd.DataFrame({
    "goal_first_count": all_counts,
    "MonitoringAgent_mean": monitoring_aligned.values,
    "GoalFirstAgent_mean": goal_first_aligned.values
})
comparison_df["Difference"] = (
    comparison_df["MonitoringAgent_mean"] -
    comparison_df["GoalFirstAgent_mean"]
)
comparison_df["Monitoring_higher"] = comparison_df["Difference"] > 0

print("\nDistinct Agents Interacted as R Comparison by Persona:")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)

# Count conditions
monitoring_higher_count = comparison_df["Monitoring_higher"].sum()
goal_first_higher_count = len(comparison_df) - monitoring_higher_count

print(f"\nMonitoringAgent higher in {monitoring_higher_count}/"
      f"{len(comparison_df)} conditions")
print(f"GoalFirstAgent higher in {goal_first_higher_count}/"
      f"{len(comparison_df)} conditions")

# ----------------------------------------------------------------------
# 3. Compare overall means
# ----------------------------------------------------------------------

overall_va = df_va_distinct_r["Number of Distinct Agents Interacted as R"].mean()
overall_ta = df_ta_distinct_r["Number of Distinct Agents Interacted as R"].mean()

print("\nOverall mean Distinct Agents Interacted as R:")
print(f"  MonitoringAgent: {overall_va:.4f}")
print(f"  GoalFirstAgent:  {overall_ta:.4f}")
print(f"  Difference:      {overall_va - overall_ta:.4f}")

# ----------------------------------------------------------------------
# 4. Mann–Whitney U tests (non-parametric)
# ----------------------------------------------------------------------

from scipy.stats import mannwhitneyu

print("\nMann–Whitney U Test (Distinct Agents Interacted as R):")
distinct_va = df_va_distinct_r["Number of Distinct Agents Interacted as R"]
distinct_ta = df_ta_distinct_r["Number of Distinct Agents Interacted as R"]

u_twosided, p_twosided = mannwhitneyu(distinct_va, distinct_ta, alternative="two-sided")
u_va_greater, p_va_greater = mannwhitneyu(distinct_va, distinct_ta, alternative="greater")
u_ta_greater, p_ta_greater = mannwhitneyu(distinct_ta, distinct_va, alternative="greater")

print(f"  Two-sided:           p = {p_twosided:.3e}")
print(f"  VA > TA (one-sided): p = {p_va_greater:.3e}")
print(f"  TA > VA (one-sided): p = {p_ta_greater:.3e}")

# Rank-biserial correlation (effect size) using two-sided U
n_va = len(distinct_va)
n_ta = len(distinct_ta)
if n_va > 0 and n_ta > 0:
    r_rb_abs = 1 - (2 * u_twosided) / (n_va * n_ta)
    # Give direction based on overall means
    direction = 1 if overall_va > overall_ta else -1
    r_rb = direction * abs(r_rb_abs)
    print(f"  Rank-biserial r:    r_rb = {r_rb:.4f} (|r| = {abs(r_rb_abs):.4f})")

if p_twosided < 0.05:
    if overall_va > overall_ta:
        print("  → SIGNIFICANT: MonitoringAgent has higher distinct agents interacted as R.")
    else:
        print("  → SIGNIFICANT: GoalFirstAgent has higher distinct agents interacted as R.")
else:
    print("  → NO significant difference.")




Distinct Agents Interacted as R Comparison by Persona:
 goal_first_count  MonitoringAgent_mean  GoalFirstAgent_mean  Difference  Monitoring_higher
                0             10.121800             0.000000   10.121800               True
              100             10.278556             7.762000    2.516556               True
              200             10.453000             7.797500    2.655500               True
              300             10.603429             7.751000    2.852429               True
              400             10.883167             7.681750    3.201417               True
              500             11.175600             7.590200    3.585400               True
              600             11.620750             7.394333    4.226417               True
              700             12.323000             6.939286    5.383714               True
              800             13.619500             6.273875    7.345625               True
              900       

# Cumulative Graphs, Wilcoxon Test:

Even if the difference of Accomplished Goals for VAs and TAs might be small, it can be consistent.
Wilcoxon test shows this.

In [21]:


df = all_agent_values_df_all_steps[["seed", "agent_config", "goal_first_count", "Step",  "AgentID", "Agent Persona", "Accomplished Goals"]]

results = []

for r in sorted(df["goal_first_count"].unique()):

    df_r = df[df["goal_first_count"] == r]

    # Compute per-seed average accomplished goals per agent
    va_seed_means = (
        df_r[df_r["Agent Persona"] == "MonitoringAgent"]
        .groupby("seed")["Accomplished Goals"]
        .mean()
    )

    ta_seed_means = (
        df_r[df_r["Agent Persona"] == "GoalFirstAgent"]
        .groupby("seed")["Accomplished Goals"]
        .mean()
    )

    # Ensure paired samples (same seeds)
    common_seeds = va_seed_means.index.intersection(ta_seed_means.index)

    va_vals = va_seed_means.loc[common_seeds]
    ta_vals = ta_seed_means.loc[common_seeds]

    # Wilcoxon signed-rank test
    stat, p = wilcoxon(ta_vals, va_vals)

    # Store results
    results.append({
        "goal_first_count": r,
        "wilcoxon_stat": stat,
        "p_value": p,
        "TA_mean": ta_vals.mean(),
        "VA_mean": va_vals.mean(),
    })

results_df = pd.DataFrame(results)
print(results_df)

/Users/efeonal/py_envs/MESA_thesis/venv/lib/python3.12/site-packages/scipy/_lib/_util.py:999: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  return fun(*args, **kwargs)


    goal_first_count  wilcoxon_stat   p_value   TA_mean   VA_mean
0                  0            NaN       NaN       NaN       NaN
1                100            0.0  0.001953  1.587843  1.763314
2                200            0.0  0.001953  1.641795  1.783585
3                300            0.0  0.001953  1.766426  1.892761
4                400            0.0  0.001953  1.745672  1.852401
5                500            0.0  0.001953  1.761690  1.849753
6                600            0.0  0.001953  1.660041  1.742406
7                700            1.0  0.003906  1.484054  1.559571
8                800            1.0  0.003906  1.145413  1.241745
9                900            3.0  0.009766  0.657194  0.723371
10              1000            NaN       NaN       NaN       NaN


/Users/efeonal/py_envs/MESA_thesis/venv/lib/python3.12/site-packages/scipy/_lib/_util.py:999: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  return fun(*args, **kwargs)


In [ ]:
# 09: Resource Conflicts per Agent Normalized by Steps — 1-way ANOVA

from scipy import stats
import numpy as np

# Start from final_agent_values (one row per agent at final step per run)
df_rc = final_agent_values.copy()

# Normalize by steps
df_rc["resource_conflicts_norm"] = df_rc["Resource Conflicts"] / df_rc["Step"]

# Split by persona
df_va_rc = df_rc[df_rc["Agent Persona"] == "MonitoringAgent"]
df_ta_rc = df_rc[df_rc["Agent Persona"] == "GoalFirstAgent"]

# ---- MonitoringAgent (VA) ----
groups_va_rc = [
    df_va_rc[df_va_rc["goal_first_count"] == r]["resource_conflicts_norm"].values
    for r in sorted(df_va_rc["goal_first_count"].unique())
]

F_va_rc, p_va_rc = stats.f_oneway(*groups_va_rc)

k_va = len(groups_va_rc)
ns_va = [len(g) for g in groups_va_rc]
N_va = sum(ns_va)
df_between_va = k_va - 1
df_within_va = N_va - k_va
eta_sq_va_rc = (F_va_rc * df_between_va) / (F_va_rc * df_between_va + df_within_va)

max_rc_va = df_va_rc.groupby("goal_first_count")["resource_conflicts_norm"].mean().max()
min_rc_va = df_va_rc.groupby("goal_first_count")["resource_conflicts_norm"].mean().min()

print("Resource Conflicts/Step — MonitoringAgent (VA)")
print(f"  F: {F_va_rc:.4f}, p: {p_va_rc:.3e}, eta^2: {eta_sq_va_rc:.4f}")
print(f"  Max mean: {max_rc_va:.4f}, Min mean: {min_rc_va:.4f}\n")

# ---- GoalFirstAgent (TA) ----
groups_ta_rc = [
    df_ta_rc[df_ta_rc["goal_first_count"] == r]["resource_conflicts_norm"].values
    for r in sorted(df_ta_rc["goal_first_count"].unique())
]

F_ta_rc, p_ta_rc = stats.f_oneway(*groups_ta_rc)

k_ta = len(groups_ta_rc)
ns_ta = [len(g) for g in groups_ta_rc]
N_ta = sum(ns_ta)
df_between_ta = k_ta - 1
df_within_ta = N_ta - k_ta
eta_sq_ta_rc = (F_ta_rc * df_between_ta) / (F_ta_rc * df_between_ta + df_within_ta)

max_rc_ta = df_ta_rc.groupby("goal_first_count")["resource_conflicts_norm"].mean().max()
min_rc_ta = df_ta_rc.groupby("goal_first_count")["resource_conflicts_norm"].mean().min()

print("Resource Conflicts/Step — GoalFirstAgent (TA)")
print(f"  F: {F_ta_rc:.4f}, p: {p_ta_rc:.3e}, eta^2: {eta_sq_ta_rc:.4f}")
print(f"  Max mean: {max_rc_ta:.4f}, Min mean: {min_rc_ta:.4f}")



In [ ]:
# 10: Counter Goal Accomplishments per Agent Normalized by Steps — 1-way ANOVA

# Start from final_agent_values (one row per agent at final step per run)
df_cg = final_agent_values.copy()

# Normalize by steps
df_cg["counter_goals_norm"] = (
    df_cg["Counter Conflict Goal Accomplishments"] / df_cg["Step"]
)

# Split by persona
df_va_cg = df_cg[df_cg["Agent Persona"] == "MonitoringAgent"]
df_ta_cg = df_cg[df_cg["Agent Persona"] == "GoalFirstAgent"]

# ---- MonitoringAgent (VA) ----
groups_va_cg = [
    df_va_cg[df_va_cg["goal_first_count"] == r]["counter_goals_norm"].values
    for r in sorted(df_va_cg["goal_first_count"].unique())
]

F_va_cg, p_va_cg = stats.f_oneway(*groups_va_cg)

k_va = len(groups_va_cg)
ns_va = [len(g) for g in groups_va_cg]
N_va = sum(ns_va)
df_between_va = k_va - 1
df_within_va = N_va - k_va
eta_sq_va_cg = (F_va_cg * df_between_va) / (F_va_cg * df_between_va + df_within_va)

max_cg_va = df_va_cg.groupby("goal_first_count")["counter_goals_norm"].mean().max()
min_cg_va = df_va_cg.groupby("goal_first_count")["counter_goals_norm"].mean().min()

print("Counter Goals/Step — MonitoringAgent (VA)")
print(f"  F: {F_va_cg:.4f}, p: {p_va_cg:.3e}, eta^2: {eta_sq_va_cg:.4f}")
print(f"  Max mean: {max_cg_va:.4f}, Min mean: {min_cg_va:.4f}\n")

# ---- GoalFirstAgent (TA) ----
groups_ta_cg = [
    df_ta_cg[df_ta_cg["goal_first_count"] == r]["counter_goals_norm"].values
    for r in sorted(df_ta_cg["goal_first_count"].unique())
]

F_ta_cg, p_ta_cg = stats.f_oneway(*groups_ta_cg)

k_ta = len(groups_ta_cg)
ns_ta = [len(g) for g in groups_ta_cg]
N_ta = sum(ns_ta)
df_between_ta = k_ta - 1
df_within_ta = N_ta - k_ta
eta_sq_ta_cg = (F_ta_cg * df_between_ta) / (F_ta_cg * df_between_ta + df_within_ta)

max_cg_ta = df_ta_cg.groupby("goal_first_count")["counter_goals_norm"].mean().max()
min_cg_ta = df_ta_cg.groupby("goal_first_count")["counter_goals_norm"].mean().min()

print("Counter Goals/Step — GoalFirstAgent (TA)")
print(f"  F: {F_ta_cg:.4f}, p: {p_ta_cg:.3e}, eta^2: {eta_sq_ta_cg:.4f}")
print(f"  Max mean: {max_cg_ta:.4f}, Min mean: {min_cg_ta:.4f}")



In [18]:
# 11: Resource Conflicts/Step Mann–Whitney U Test (VA vs TA)

from scipy.stats import mannwhitneyu
import pandas as pd

# ----------------------------------------------------------------------
# 1. Prepare data properly: aggregate per run (like df_da_full pattern)
# ----------------------------------------------------------------------

df_all_agents = final_agent_values.copy()

# Get simulation's total steps per run
sim_steps_per_run = (
    df_all_agents.groupby(["goal_first_count", "seed"])["Step"]
    .max()
    .reset_index()
    .rename(columns={"Step": "sim_total_steps"})
)

# Aggregate to run-level: average Resource Conflicts per agent type per run
run_level_df = (
    df_all_agents.groupby(["goal_first_count", "seed", "Agent Persona"])[["Resource Conflicts"]]
    .mean()
    .reset_index()
)

# Merge simulation steps
run_level_df = run_level_df.merge(sim_steps_per_run, on=["goal_first_count", "seed"])

# Normalize by simulation's total steps (matching graph: avg_resource_conflicts / avg_steps_overall)
run_level_df["resource_conflicts_norm"] = (
    run_level_df["Resource Conflicts"] / run_level_df["sim_total_steps"]
)

# Split by persona
df_va = run_level_df[run_level_df["Agent Persona"] == "MonitoringAgent"]
df_ta = run_level_df[run_level_df["Agent Persona"] == "GoalFirstAgent"]

# Per-condition means (for a quick descriptive table)
va_means = df_va.groupby("goal_first_count")["resource_conflicts_norm"].mean()
ta_means = df_ta.groupby("goal_first_count")["resource_conflicts_norm"].mean()

all_counts = sorted(set(va_means.index).union(set(ta_means.index)))
va_aligned = va_means.reindex(all_counts, fill_value=0)
ta_aligned = ta_means.reindex(all_counts, fill_value=0)

comparison_df = pd.DataFrame({
    "goal_first_count": all_counts,
    "MonitoringAgent_mean": va_aligned.values,
    "GoalFirstAgent_mean": ta_aligned.values,
})
comparison_df["Difference"] = comparison_df["MonitoringAgent_mean"] - comparison_df["GoalFirstAgent_mean"]
comparison_df["Monitoring_higher"] = comparison_df["Difference"] > 0

print("\nResource Conflicts/Step Comparison by Persona:")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)

monitoring_higher_count = comparison_df["Monitoring_higher"].sum()
goal_first_higher_count = len(comparison_df) - monitoring_higher_count
print(f"\nMonitoringAgent higher in {monitoring_higher_count}/{len(comparison_df)} conditions")
print(f"GoalFirstAgent higher in {goal_first_higher_count}/{len(comparison_df)} conditions")

# Global Mann–Whitney U test on run-level values (all seeds × configs)
print("\nMann–Whitney U Test (Resource Conflicts/Step):")
cons_va = df_va["resource_conflicts_norm"]
cons_ta = df_ta["resource_conflicts_norm"]

u_twosided, p_twosided = mannwhitneyu(cons_va, cons_ta, alternative="two-sided")
u_va_greater, p_va_greater = mannwhitneyu(cons_va, cons_ta, alternative="greater")
u_ta_greater, p_ta_greater = mannwhitneyu(cons_ta, cons_va, alternative="greater")

print(f"  Two-sided:           p = {p_twosided:.3e}")
print(f"  VA > TA (one-sided): p = {p_va_greater:.3e}")
print(f"  TA > VA (one-sided): p = {p_ta_greater:.3e}")

mean_va = cons_va.mean()
mean_ta = cons_ta.mean()
print(f"  MonitoringAgent mean RC/step: {mean_va:.4f}")
print(f"  GoalFirstAgent    mean RC/step: {mean_ta:.4f}")
print(f"  Difference:                     {mean_va - mean_ta:.4f}")

n_va = len(cons_va)
n_ta = len(cons_ta)
if n_va > 0 and n_ta > 0:
    r_rb_abs = 1 - (2 * u_twosided) / (n_va * n_ta)
    direction = 1 if mean_va > mean_ta else -1
    r_rb = direction * abs(r_rb_abs)
    print(f"  Rank-biserial r:    r_rb = {r_rb:.4f} (|r| = {abs(r_rb_abs):.4f})")

if p_twosided < 0.05:
    if mean_va > mean_ta:
        print("  → SIGNIFICANT: MonitoringAgent has higher RC/step.")
    else:
        print("  → SIGNIFICANT: GoalFirstAgent has higher RC/step.")
else:
    print("  → NO significant difference in RC/step.")




Resource Conflicts/Step Comparison by Persona:
 goal_first_count  MonitoringAgent_mean  GoalFirstAgent_mean  Difference  Monitoring_higher
                0              0.042344             0.000000    0.042344               True
              100              0.077221             0.153297   -0.076076              False
              200              0.117534             0.180483   -0.062948              False
              300              0.145487             0.194327   -0.048840              False
              400              0.195881             0.238748   -0.042867              False
              500              0.242974             0.282175   -0.039202              False
              600              0.323584             0.358963   -0.035379              False
              700              0.447177             0.477762   -0.030584              False
              800              0.599839             0.638214   -0.038375              False
              900              0

In [19]:
# 12: Counter Conflict Goals/Step Mann–Whitney U Test (VA vs TA)

from scipy.stats import mannwhitneyu
import pandas as pd

# ----------------------------------------------------------------------
# 1. Prepare data properly: aggregate per run (like df_da_full pattern)
# ----------------------------------------------------------------------

df_all_agents = final_agent_values.copy()

# Get simulation's total steps per run
sim_steps_per_run = (
    df_all_agents.groupby(["goal_first_count", "seed"])["Step"]
    .max()
    .reset_index()
    .rename(columns={"Step": "sim_total_steps"})
)

# Aggregate to run-level: average Counter Goal Accomplishments per agent type per run
run_level_df = (
    df_all_agents.groupby(["goal_first_count", "seed", "Agent Persona"])[["Counter Conflict Goal Accomplishments"]]
    .mean()
    .reset_index()
)

# Merge simulation steps
run_level_df = run_level_df.merge(sim_steps_per_run, on=["goal_first_count", "seed"])

# Normalize by simulation's total steps (matching graph: avg_counter_goals / avg_steps_overall)
run_level_df["counter_goals_norm"] = (
    run_level_df["Counter Conflict Goal Accomplishments"] / run_level_df["sim_total_steps"]
)

# Split by persona
df_va = run_level_df[run_level_df["Agent Persona"] == "MonitoringAgent"]
df_ta = run_level_df[run_level_df["Agent Persona"] == "GoalFirstAgent"]

# Per-condition means (for a quick descriptive table)
va_means = df_va.groupby("goal_first_count")["counter_goals_norm"].mean()
ta_means = df_ta.groupby("goal_first_count")["counter_goals_norm"].mean()

all_counts = sorted(set(va_means.index).union(set(ta_means.index)))
va_aligned = va_means.reindex(all_counts, fill_value=0)
ta_aligned = ta_means.reindex(all_counts, fill_value=0)

comparison_df = pd.DataFrame({
    "goal_first_count": all_counts,
    "MonitoringAgent_mean": va_aligned.values,
    "GoalFirstAgent_mean": ta_aligned.values,
})
comparison_df["Difference"] = comparison_df["MonitoringAgent_mean"] - comparison_df["GoalFirstAgent_mean"]
comparison_df["Monitoring_higher"] = comparison_df["Difference"] > 0

print("\nCounter Goals/Step Comparison by Persona:")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)

monitoring_higher_count = comparison_df["Monitoring_higher"].sum()
goal_first_higher_count = len(comparison_df) - monitoring_higher_count
print(f"\nMonitoringAgent higher in {monitoring_higher_count}/{len(comparison_df)} conditions")
print(f"GoalFirstAgent higher in {goal_first_higher_count}/{len(comparison_df)} conditions")

# Global Mann–Whitney U test on run-level values (all seeds × configs)
print("\nMann–Whitney U Test (Counter Conflict Goals/Step):")
cons_va = df_va["counter_goals_norm"]
cons_ta = df_ta["counter_goals_norm"]

u_twosided, p_twosided = mannwhitneyu(cons_va, cons_ta, alternative="two-sided")
u_va_greater, p_va_greater = mannwhitneyu(cons_va, cons_ta, alternative="greater")
u_ta_greater, p_ta_greater = mannwhitneyu(cons_ta, cons_va, alternative="greater")

print(f"  Two-sided:           p = {p_twosided:.3e}")
print(f"  VA > TA (one-sided): p = {p_va_greater:.3e}")
print(f"  TA > VA (one-sided): p = {p_ta_greater:.3e}")

mean_va = cons_va.mean()
mean_ta = cons_ta.mean()
print(f"  MonitoringAgent mean counter-goals/step: {mean_va:.4f}")
print(f"  GoalFirstAgent    mean counter-goals/step: {mean_ta:.4f}")
print(f"  Difference:                                 {mean_va - mean_ta:.4f}")

n_va = len(cons_va)
n_ta = len(cons_ta)
if n_va > 0 and n_ta > 0:
    r_rb_abs = 1 - (2 * u_twosided) / (n_va * n_ta)
    direction = 1 if mean_va > mean_ta else -1
    r_rb = direction * abs(r_rb_abs)
    print(f"  Rank-biserial r:    r_rb = {r_rb:.4f} (|r| = {abs(r_rb_abs):.4f})")

if p_twosided < 0.05:
    if mean_va > mean_ta:
        print("  → SIGNIFICANT: MonitoringAgent has higher counter-goals/step.")
    else:
        print("  → SIGNIFICANT: GoalFirstAgent has higher counter-goals/step.")
else:
    print("  → NO significant difference in counter-goals/step.")




Counter Goals/Step Comparison by Persona:
 goal_first_count  MonitoringAgent_mean  GoalFirstAgent_mean  Difference  Monitoring_higher
                0              0.014471             0.000000    0.014471               True
              100              0.014322             0.079176   -0.064854              False
              200              0.014895             0.071605   -0.056709              False
              300              0.014097             0.059196   -0.045099              False
              400              0.012925             0.052931   -0.040006              False
              500              0.012077             0.041899   -0.029821              False
              600              0.010260             0.033521   -0.023261              False
              700              0.007204             0.021340   -0.014136              False
              800              0.004033             0.009939   -0.005906              False
              900              0.0011